# Evaluasi Retrieval dan RAGAS

Jalankan seluruh sel secara berurutan di VS Code. RAGAS mengevaluasi 56 pertanyaan dengan jawaban menggunakan enam konteks retrieval teratas.
Pengujian pertama memerlukan koneksi API. Pengujian berikutnya melanjutkan checkpoint yang sama.
Ubah `ULANG_RAGAS` menjadi `True` untuk membuat pengujian baru.


In [1]:
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from IPython.display import Markdown, display

ULANG_RAGAS = False
RAGAS_RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
PYTHON_EVALUASI = ""

PROJECT = next(
    candidate
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    for candidate in (parent, parent / "Program")
    if (candidate / "config.py").is_file()
    and (candidate / "evaluation/notebook_evaluation.py").is_file()
)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from evaluation.notebook_evaluation import (
    markdown_table, ragas_context_table, ragas_tables, read_json, read_jsonl,
    retrieval_question_tables, retrieval_tables, write_csv,
)

DATASET = PROJECT / "data/evaluation/thesis_benchmark"
HASIL = PROJECT / "evaluation/results/evaluasi_retrieval_ragas"
BUNDLED_PYTHON = Path.home() / ".cache/codex-runtimes/codex-primary-runtime/dependencies/python/python.exe"
candidates = [Path(PYTHON_EVALUASI)] if PYTHON_EVALUASI else [
    PROJECT / ".venv/Scripts/python.exe", PROJECT / ".venv/bin/python",
    Path(sys.executable), BUNDLED_PYTHON,
]
for candidate in dict.fromkeys(candidates):
    if not candidate.is_file():
        continue
    environment = os.environ.copy()
    environment.pop("PYTHONPATH", None)
    environment["PYTHONUTF8"] = "1"
    environment["RAGAS_DO_NOT_TRACK"] = "true"
    environment["ANONYMIZED_TELEMETRY"] = "false"
    if candidate.resolve() == BUNDLED_PYTHON.resolve():
        environment["PYTHONPATH"] = os.pathsep.join(
            str(PROJECT / name) for name in (".evaluation-deps", ".python-deps")
            if (PROJECT / name).is_dir()
        )
    probe = subprocess.run(
        [str(candidate), "-c", "import sys, importlib.util; print(sys.version_info[:2] == (3, 12) and all(importlib.util.find_spec(m) for m in ('numpy', 'chromadb', 'rank_bm25', 'ragas', 'instructor', 'openai')))"] ,
        cwd=PROJECT, env=environment, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=15,
    )
    if probe.returncode == 0 and probe.stdout.strip() == "True":
        PYTHON, ENVIRONMENT = candidate, environment
        break
else:
    raise RuntimeError("Gunakan Python 3.12 dengan requirements.txt dan requirements-evaluation.txt. Isi PYTHON_EVALUASI jika interpreter berada di lokasi lain.")

questions = read_jsonl(DATASET / "questions.jsonl")
references = read_jsonl(DATASET / "ground_truth.jsonl")
registry = read_json(PROJECT / "data/registry/index_registry.json")
HASIL.mkdir(parents=True, exist_ok=True)

def jalankan(*arguments):
    result = subprocess.run(
        [str(PYTHON), "-X", "utf8", "-u", *map(str, arguments)],
        cwd=PROJECT, env=ENVIRONMENT, capture_output=True, text=True,
        encoding="utf-8", errors="replace",
    )
    if result.returncode:
        message = "\n".join((result.stdout + "\n" + result.stderr).splitlines()[-20:])
        raise RuntimeError(message)


## Hasil Pengujian Retrieval

In [2]:
jalankan("-m", "evaluation.notebook_evaluation", "--output", HASIL / "retrieval.json")
retrieval = read_json(HASIL / "retrieval.json")
distribusi, keseluruhan, kategori = retrieval_tables(questions, references, registry, retrieval)


### Tabel L12.1 Distribusi Dokumen dan Pertanyaan Pengujian Retrieval

In [3]:
display(Markdown(markdown_table(distribusi)))


| Emiten | Jumlah chunk | Pertanyaan dengan jawaban | Pertanyaan tanpa jawaban |
| --- | --- | --- | --- |
| BACH | 2.026 | 8 | 1 |
| EMMI | 1.730 | 8 | 1 |
| JECX | 2.616 | 8 | 1 |
| JELI | 1.813 | 8 | 1 |
| PRDL | 1.908 | 8 | 1 |
| RANS | 3.007 | 8 | 1 |
| SWAP | 2.599 | 8 | 1 |
| Total | 15.699 | 56 | 7 |

### Tabel L12.2 Hasil Pengujian Retrieval Keseluruhan

In [4]:
display(Markdown(markdown_table(keseluruhan)))


| Metode | K | Precision@K | Recall@K | MRR@K |
| --- | --- | --- | --- | --- |
| Sparse Retrieval | 1 | 0,3750 | 0,0405 | 0,3750 |
| Sparse Retrieval | 3 | 0,3095 | 0,1055 | 0,4851 |
| Sparse Retrieval | 6 | 0,2232 | 0,1550 | 0,5024 |
| Dense Retrieval | 1 | 0,4643 | 0,0652 | 0,4643 |
| Dense Retrieval | 3 | 0,3571 | 0,1361 | 0,5863 |
| Dense Retrieval | 6 | 0,2589 | 0,1919 | 0,5943 |
| Hybrid Retrieval with RRF | 1 | 0,4107 | 0,0548 | 0,4107 |
| Hybrid Retrieval with RRF | 3 | 0,3631 | 0,1389 | 0,5446 |
| Hybrid Retrieval with RRF | 6 | 0,3065 | 0,2185 | 0,5780 |

### Tabel L12.3 Hasil Retrieval Berdasarkan Kategori Pertanyaan pada K = 6

In [5]:
display(Markdown(markdown_table(kategori)))


| Kategori | Metode | Precision@6 | Recall@6 | MRR@6 |
| --- | --- | --- | --- | --- |
| Fakta eksplisit | Sparse | 0,3214 | 0,2532 | 0,6929 |
| Fakta eksplisit | Dense | 0,3095 | 0,2448 | 0,7857 |
| Fakta eksplisit | Hybrid | 0,4405 | 0,3522 | 0,7917 |
| Parafrasa | Sparse | 0,2024 | 0,0939 | 0,5238 |
| Parafrasa | Dense | 0,2024 | 0,1498 | 0,5893 |
| Parafrasa | Hybrid | 0,2857 | 0,1826 | 0,5833 |
| Tabel numerik | Sparse | 0,2500 | 0,2208 | 0,5167 |
| Tabel numerik | Dense | 0,3095 | 0,2782 | 0,6214 |
| Tabel numerik | Hybrid | 0,2976 | 0,2507 | 0,6214 |
| Lintas bagian | Sparse | 0,1190 | 0,0521 | 0,2762 |
| Lintas bagian | Dense | 0,2143 | 0,0947 | 0,3810 |
| Lintas bagian | Hybrid | 0,2024 | 0,0884 | 0,3155 |

### Hasil Pengujian Retrieval per Pertanyaan

In [6]:
teks_chunk = read_json(HASIL / "retrieval_chunks.json")
rincian_retrieval, peringkat_retrieval = retrieval_question_tables(questions, references, retrieval, teks_chunk)
kolom_retrieval = ["No", "Emiten", "Pertanyaan", "Metode", "K", "Precision@K", "Recall@K", "MRR@K"]
nama_metode = {"Sparse Retrieval": "Sparse", "Dense Retrieval": "Dense", "Hybrid Retrieval with RRF": "Hybrid"}
tabel_retrieval = [
    {kolom: nama_metode.get(row[kolom], row[kolom]) if kolom == "Metode" else row[kolom]
     for kolom in kolom_retrieval}
    for row in rincian_retrieval
]
display(Markdown(markdown_table(tabel_retrieval)))


| No | Emiten | Pertanyaan | Metode | K | Precision@K | Recall@K | MRR@K |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | BACH | Berapa harga penawaran saham BACH pada prospektus ini? | Sparse | 1 | 1,0000 | 0,1111 | 1,0000 |
| 1 | BACH | Berapa harga penawaran saham BACH pada prospektus ini? | Sparse | 3 | 0,3333 | 0,1111 | 1,0000 |
| 1 | BACH | Berapa harga penawaran saham BACH pada prospektus ini? | Sparse | 6 | 0,3333 | 0,2222 | 1,0000 |
| 1 | BACH | Berapa harga penawaran saham BACH pada prospektus ini? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 1 | BACH | Berapa harga penawaran saham BACH pada prospektus ini? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 1 | BACH | Berapa harga penawaran saham BACH pada prospektus ini? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 1 | BACH | Berapa harga penawaran saham BACH pada prospektus ini? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 1 | BACH | Berapa harga penawaran saham BACH pada prospektus ini? | Hybrid | 3 | 0,3333 | 0,1111 | 0,5000 |
| 1 | BACH | Berapa harga penawaran saham BACH pada prospektus ini? | Hybrid | 6 | 0,1667 | 0,1111 | 0,5000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek IPO BACH? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek IPO BACH? | Sparse | 3 | 0,3333 | 0,1111 | 0,5000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek IPO BACH? | Sparse | 6 | 0,1667 | 0,1111 | 0,5000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek IPO BACH? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek IPO BACH? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek IPO BACH? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek IPO BACH? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek IPO BACH? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek IPO BACH? | Hybrid | 6 | 0,3333 | 0,2222 | 0,2500 |
| 3 | BACH | Bisnis utama BACH bergerak di bidang apa? | Sparse | 1 | 1,0000 | 0,0208 | 1,0000 |
| 3 | BACH | Bisnis utama BACH bergerak di bidang apa? | Sparse | 3 | 0,6667 | 0,0417 | 1,0000 |
| 3 | BACH | Bisnis utama BACH bergerak di bidang apa? | Sparse | 6 | 0,6667 | 0,0833 | 1,0000 |
| 3 | BACH | Bisnis utama BACH bergerak di bidang apa? | Dense | 1 | 1,0000 | 0,0208 | 1,0000 |
| 3 | BACH | Bisnis utama BACH bergerak di bidang apa? | Dense | 3 | 0,3333 | 0,0208 | 1,0000 |
| 3 | BACH | Bisnis utama BACH bergerak di bidang apa? | Dense | 6 | 0,1667 | 0,0208 | 1,0000 |
| 3 | BACH | Bisnis utama BACH bergerak di bidang apa? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 3 | BACH | Bisnis utama BACH bergerak di bidang apa? | Hybrid | 3 | 0,6667 | 0,0417 | 0,5000 |
| 3 | BACH | Bisnis utama BACH bergerak di bidang apa? | Hybrid | 6 | 0,5000 | 0,0625 | 0,5000 |
| 4 | BACH | Apa ancaman bisnis yang disebut paling utama bagi BACH? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 4 | BACH | Apa ancaman bisnis yang disebut paling utama bagi BACH? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 4 | BACH | Apa ancaman bisnis yang disebut paling utama bagi BACH? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 4 | BACH | Apa ancaman bisnis yang disebut paling utama bagi BACH? | Dense | 1 | 1,0000 | 0,2500 | 1,0000 |
| 4 | BACH | Apa ancaman bisnis yang disebut paling utama bagi BACH? | Dense | 3 | 0,3333 | 0,2500 | 1,0000 |
| 4 | BACH | Apa ancaman bisnis yang disebut paling utama bagi BACH? | Dense | 6 | 0,1667 | 0,2500 | 1,0000 |
| 4 | BACH | Apa ancaman bisnis yang disebut paling utama bagi BACH? | Hybrid | 1 | 1,0000 | 0,2500 | 1,0000 |
| 4 | BACH | Apa ancaman bisnis yang disebut paling utama bagi BACH? | Hybrid | 3 | 0,3333 | 0,2500 | 1,0000 |
| 4 | BACH | Apa ancaman bisnis yang disebut paling utama bagi BACH? | Hybrid | 6 | 0,1667 | 0,2500 | 1,0000 |
| 5 | BACH | Berapa total aset konsolidasian BACH per 31 Desember 2025 dalam Rupiah? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 5 | BACH | Berapa total aset konsolidasian BACH per 31 Desember 2025 dalam Rupiah? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 5 | BACH | Berapa total aset konsolidasian BACH per 31 Desember 2025 dalam Rupiah? | Sparse | 6 | 0,1667 | 0,1667 | 0,2000 |
| 5 | BACH | Berapa total aset konsolidasian BACH per 31 Desember 2025 dalam Rupiah? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 5 | BACH | Berapa total aset konsolidasian BACH per 31 Desember 2025 dalam Rupiah? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 5 | BACH | Berapa total aset konsolidasian BACH per 31 Desember 2025 dalam Rupiah? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 5 | BACH | Berapa total aset konsolidasian BACH per 31 Desember 2025 dalam Rupiah? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 5 | BACH | Berapa total aset konsolidasian BACH per 31 Desember 2025 dalam Rupiah? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 5 | BACH | Berapa total aset konsolidasian BACH per 31 Desember 2025 dalam Rupiah? | Hybrid | 6 | 0,0000 | 0,0000 | 0,0000 |
| 6 | BACH | Berapa pendapatan neto konsolidasian BACH untuk tahun 2025 dalam Rupiah? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 6 | BACH | Berapa pendapatan neto konsolidasian BACH untuk tahun 2025 dalam Rupiah? | Sparse | 3 | 0,3333 | 0,1250 | 0,5000 |
| 6 | BACH | Berapa pendapatan neto konsolidasian BACH untuk tahun 2025 dalam Rupiah? | Sparse | 6 | 0,3333 | 0,2500 | 0,5000 |
| 6 | BACH | Berapa pendapatan neto konsolidasian BACH untuk tahun 2025 dalam Rupiah? | Dense | 1 | 1,0000 | 0,1250 | 1,0000 |
| 6 | BACH | Berapa pendapatan neto konsolidasian BACH untuk tahun 2025 dalam Rupiah? | Dense | 3 | 0,6667 | 0,2500 | 1,0000 |
| 6 | BACH | Berapa pendapatan neto konsolidasian BACH untuk tahun 2025 dalam Rupiah? | Dense | 6 | 0,5000 | 0,3750 | 1,0000 |
| 6 | BACH | Berapa pendapatan neto konsolidasian BACH untuk tahun 2025 dalam Rupiah? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 6 | BACH | Berapa pendapatan neto konsolidasian BACH untuk tahun 2025 dalam Rupiah? | Hybrid | 3 | 0,6667 | 0,2500 | 0,5000 |
| 6 | BACH | Berapa pendapatan neto konsolidasian BACH untuk tahun 2025 dalam Rupiah? | Hybrid | 6 | 0,5000 | 0,3750 | 0,5000 |
| 7 | BACH | Jelaskan bidang usaha utama BACH dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 7 | BACH | Jelaskan bidang usaha utama BACH dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 7 | BACH | Jelaskan bidang usaha utama BACH dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 6 | 0,1667 | 0,0185 | 0,2000 |
| 7 | BACH | Jelaskan bidang usaha utama BACH dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 7 | BACH | Jelaskan bidang usaha utama BACH dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 7 | BACH | Jelaskan bidang usaha utama BACH dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 7 | BACH | Jelaskan bidang usaha utama BACH dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 7 | BACH | Jelaskan bidang usaha utama BACH dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 7 | BACH | Jelaskan bidang usaha utama BACH dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 6 | 0,0000 | 0,0000 | 0,0000 |
| 8 | BACH | Apa risiko utama BACH, dan berapa pendapatan neto tahun 2025 dalam Rupiah pada laporan konsolidasian? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 8 | BACH | Apa risiko utama BACH, dan berapa pendapatan neto tahun 2025 dalam Rupiah pada laporan konsolidasian? | Sparse | 3 | 0,3333 | 0,0833 | 0,3333 |
| 8 | BACH | Apa risiko utama BACH, dan berapa pendapatan neto tahun 2025 dalam Rupiah pada laporan konsolidasian? | Sparse | 6 | 0,3333 | 0,1667 | 0,3333 |
| 8 | BACH | Apa risiko utama BACH, dan berapa pendapatan neto tahun 2025 dalam Rupiah pada laporan konsolidasian? | Dense | 1 | 1,0000 | 0,0833 | 1,0000 |
| 8 | BACH | Apa risiko utama BACH, dan berapa pendapatan neto tahun 2025 dalam Rupiah pada laporan konsolidasian? | Dense | 3 | 0,6667 | 0,1667 | 1,0000 |
| 8 | BACH | Apa risiko utama BACH, dan berapa pendapatan neto tahun 2025 dalam Rupiah pada laporan konsolidasian? | Dense | 6 | 0,5000 | 0,2500 | 1,0000 |
| 8 | BACH | Apa risiko utama BACH, dan berapa pendapatan neto tahun 2025 dalam Rupiah pada laporan konsolidasian? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 8 | BACH | Apa risiko utama BACH, dan berapa pendapatan neto tahun 2025 dalam Rupiah pada laporan konsolidasian? | Hybrid | 3 | 0,3333 | 0,0833 | 0,3333 |
| 8 | BACH | Apa risiko utama BACH, dan berapa pendapatan neto tahun 2025 dalam Rupiah pada laporan konsolidasian? | Hybrid | 6 | 0,5000 | 0,2500 | 0,3333 |
| 9 | EMMI | Berapa harga penawaran saham EMMI pada prospektus ini? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 9 | EMMI | Berapa harga penawaran saham EMMI pada prospektus ini? | Sparse | 3 | 0,3333 | 0,0909 | 0,5000 |
| 9 | EMMI | Berapa harga penawaran saham EMMI pada prospektus ini? | Sparse | 6 | 0,3333 | 0,1818 | 0,5000 |
| 9 | EMMI | Berapa harga penawaran saham EMMI pada prospektus ini? | Dense | 1 | 1,0000 | 0,0909 | 1,0000 |
| 9 | EMMI | Berapa harga penawaran saham EMMI pada prospektus ini? | Dense | 3 | 0,6667 | 0,1818 | 1,0000 |
| 9 | EMMI | Berapa harga penawaran saham EMMI pada prospektus ini? | Dense | 6 | 0,5000 | 0,2727 | 1,0000 |
| 9 | EMMI | Berapa harga penawaran saham EMMI pada prospektus ini? | Hybrid | 1 | 1,0000 | 0,0909 | 1,0000 |
| 9 | EMMI | Berapa harga penawaran saham EMMI pada prospektus ini? | Hybrid | 3 | 0,6667 | 0,1818 | 1,0000 |
| 9 | EMMI | Berapa harga penawaran saham EMMI pada prospektus ini? | Hybrid | 6 | 0,5000 | 0,2727 | 1,0000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek IPO EMMI? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek IPO EMMI? | Sparse | 3 | 0,6667 | 0,3333 | 0,5000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek IPO EMMI? | Sparse | 6 | 0,5000 | 0,5000 | 0,5000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek IPO EMMI? | Dense | 1 | 1,0000 | 0,1667 | 1,0000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek IPO EMMI? | Dense | 3 | 0,6667 | 0,3333 | 1,0000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek IPO EMMI? | Dense | 6 | 0,3333 | 0,3333 | 1,0000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek IPO EMMI? | Hybrid | 1 | 1,0000 | 0,1667 | 1,0000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek IPO EMMI? | Hybrid | 3 | 0,6667 | 0,3333 | 1,0000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek IPO EMMI? | Hybrid | 6 | 0,5000 | 0,5000 | 1,0000 |
| 11 | EMMI | Bisnis utama EMMI bergerak di bidang apa? | Sparse | 1 | 1,0000 | 0,0303 | 1,0000 |
| 11 | EMMI | Bisnis utama EMMI bergerak di bidang apa? | Sparse | 3 | 0,6667 | 0,0606 | 1,0000 |
| 11 | EMMI | Bisnis utama EMMI bergerak di bidang apa? | Sparse | 6 | 0,3333 | 0,0606 | 1,0000 |
| 11 | EMMI | Bisnis utama EMMI bergerak di bidang apa? | Dense | 1 | 1,0000 | 0,0303 | 1,0000 |
| 11 | EMMI | Bisnis utama EMMI bergerak di bidang apa? | Dense | 3 | 0,6667 | 0,0606 | 1,0000 |
| 11 | EMMI | Bisnis utama EMMI bergerak di bidang apa? | Dense | 6 | 0,6667 | 0,1212 | 1,0000 |
| 11 | EMMI | Bisnis utama EMMI bergerak di bidang apa? | Hybrid | 1 | 1,0000 | 0,0303 | 1,0000 |
| 11 | EMMI | Bisnis utama EMMI bergerak di bidang apa? | Hybrid | 3 | 0,6667 | 0,0606 | 1,0000 |
| 11 | EMMI | Bisnis utama EMMI bergerak di bidang apa? | Hybrid | 6 | 0,6667 | 0,1212 | 1,0000 |
| 12 | EMMI | Apa ancaman bisnis yang disebut paling utama bagi EMMI? | Sparse | 1 | 1,0000 | 0,1429 | 1,0000 |
| 12 | EMMI | Apa ancaman bisnis yang disebut paling utama bagi EMMI? | Sparse | 3 | 0,3333 | 0,1429 | 1,0000 |
| 12 | EMMI | Apa ancaman bisnis yang disebut paling utama bagi EMMI? | Sparse | 6 | 0,1667 | 0,1429 | 1,0000 |
| 12 | EMMI | Apa ancaman bisnis yang disebut paling utama bagi EMMI? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 12 | EMMI | Apa ancaman bisnis yang disebut paling utama bagi EMMI? | Dense | 3 | 0,3333 | 0,1429 | 0,5000 |
| 12 | EMMI | Apa ancaman bisnis yang disebut paling utama bagi EMMI? | Dense | 6 | 0,3333 | 0,2857 | 0,5000 |
| 12 | EMMI | Apa ancaman bisnis yang disebut paling utama bagi EMMI? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 12 | EMMI | Apa ancaman bisnis yang disebut paling utama bagi EMMI? | Hybrid | 3 | 0,3333 | 0,1429 | 0,5000 |
| 12 | EMMI | Apa ancaman bisnis yang disebut paling utama bagi EMMI? | Hybrid | 6 | 0,3333 | 0,2857 | 0,5000 |
| 13 | EMMI | Berapa total aset konsolidasian EMMI per 31 Desember 2025 dalam Rupiah? | Sparse | 1 | 1,0000 | 0,0909 | 1,0000 |
| 13 | EMMI | Berapa total aset konsolidasian EMMI per 31 Desember 2025 dalam Rupiah? | Sparse | 3 | 0,6667 | 0,1818 | 1,0000 |
| 13 | EMMI | Berapa total aset konsolidasian EMMI per 31 Desember 2025 dalam Rupiah? | Sparse | 6 | 0,3333 | 0,1818 | 1,0000 |
| 13 | EMMI | Berapa total aset konsolidasian EMMI per 31 Desember 2025 dalam Rupiah? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 13 | EMMI | Berapa total aset konsolidasian EMMI per 31 Desember 2025 dalam Rupiah? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 13 | EMMI | Berapa total aset konsolidasian EMMI per 31 Desember 2025 dalam Rupiah? | Dense | 6 | 0,1667 | 0,0909 | 0,2000 |
| 13 | EMMI | Berapa total aset konsolidasian EMMI per 31 Desember 2025 dalam Rupiah? | Hybrid | 1 | 1,0000 | 0,0909 | 1,0000 |
| 13 | EMMI | Berapa total aset konsolidasian EMMI per 31 Desember 2025 dalam Rupiah? | Hybrid | 3 | 0,3333 | 0,0909 | 1,0000 |
| 13 | EMMI | Berapa total aset konsolidasian EMMI per 31 Desember 2025 dalam Rupiah? | Hybrid | 6 | 0,3333 | 0,1818 | 1,0000 |
| 14 | EMMI | Berapa penjualan bersih konsolidasian EMMI untuk tahun 2025 dalam Rupiah? | Sparse | 1 | 1,0000 | 0,0909 | 1,0000 |
| 14 | EMMI | Berapa penjualan bersih konsolidasian EMMI untuk tahun 2025 dalam Rupiah? | Sparse | 3 | 0,6667 | 0,1818 | 1,0000 |
| 14 | EMMI | Berapa penjualan bersih konsolidasian EMMI untuk tahun 2025 dalam Rupiah? | Sparse | 6 | 0,3333 | 0,1818 | 1,0000 |
| 14 | EMMI | Berapa penjualan bersih konsolidasian EMMI untuk tahun 2025 dalam Rupiah? | Dense | 1 | 1,0000 | 0,0909 | 1,0000 |
| 14 | EMMI | Berapa penjualan bersih konsolidasian EMMI untuk tahun 2025 dalam Rupiah? | Dense | 3 | 0,6667 | 0,1818 | 1,0000 |
| 14 | EMMI | Berapa penjualan bersih konsolidasian EMMI untuk tahun 2025 dalam Rupiah? | Dense | 6 | 0,5000 | 0,2727 | 1,0000 |
| 14 | EMMI | Berapa penjualan bersih konsolidasian EMMI untuk tahun 2025 dalam Rupiah? | Hybrid | 1 | 1,0000 | 0,0909 | 1,0000 |
| 14 | EMMI | Berapa penjualan bersih konsolidasian EMMI untuk tahun 2025 dalam Rupiah? | Hybrid | 3 | 1,0000 | 0,2727 | 1,0000 |
| 14 | EMMI | Berapa penjualan bersih konsolidasian EMMI untuk tahun 2025 dalam Rupiah? | Hybrid | 6 | 0,6667 | 0,3636 | 1,0000 |
| 15 | EMMI | Jelaskan bidang usaha utama EMMI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 15 | EMMI | Jelaskan bidang usaha utama EMMI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 15 | EMMI | Jelaskan bidang usaha utama EMMI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 15 | EMMI | Jelaskan bidang usaha utama EMMI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 15 | EMMI | Jelaskan bidang usaha utama EMMI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 3 | 0,3333 | 0,0278 | 0,5000 |
| 15 | EMMI | Jelaskan bidang usaha utama EMMI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 6 | 0,1667 | 0,0278 | 0,5000 |
| 15 | EMMI | Jelaskan bidang usaha utama EMMI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 15 | EMMI | Jelaskan bidang usaha utama EMMI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 15 | EMMI | Jelaskan bidang usaha utama EMMI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 6 | 0,1667 | 0,0278 | 0,2500 |
| 16 | EMMI | Apa risiko utama EMMI, dan berapa penjualan bersih tahun 2025 dalam Rupiah pada laporan konsolidasian? | Sparse | 1 | 1,0000 | 0,0556 | 1,0000 |
| 16 | EMMI | Apa risiko utama EMMI, dan berapa penjualan bersih tahun 2025 dalam Rupiah pada laporan konsolidasian? | Sparse | 3 | 0,3333 | 0,0556 | 1,0000 |
| 16 | EMMI | Apa risiko utama EMMI, dan berapa penjualan bersih tahun 2025 dalam Rupiah pada laporan konsolidasian? | Sparse | 6 | 0,1667 | 0,0556 | 1,0000 |
| 16 | EMMI | Apa risiko utama EMMI, dan berapa penjualan bersih tahun 2025 dalam Rupiah pada laporan konsolidasian? | Dense | 1 | 1,0000 | 0,0556 | 1,0000 |
| 16 | EMMI | Apa risiko utama EMMI, dan berapa penjualan bersih tahun 2025 dalam Rupiah pada laporan konsolidasian? | Dense | 3 | 1,0000 | 0,1667 | 1,0000 |
| 16 | EMMI | Apa risiko utama EMMI, dan berapa penjualan bersih tahun 2025 dalam Rupiah pada laporan konsolidasian? | Dense | 6 | 0,5000 | 0,1667 | 1,0000 |
| 16 | EMMI | Apa risiko utama EMMI, dan berapa penjualan bersih tahun 2025 dalam Rupiah pada laporan konsolidasian? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 16 | EMMI | Apa risiko utama EMMI, dan berapa penjualan bersih tahun 2025 dalam Rupiah pada laporan konsolidasian? | Hybrid | 3 | 0,6667 | 0,1111 | 0,5000 |
| 16 | EMMI | Apa risiko utama EMMI, dan berapa penjualan bersih tahun 2025 dalam Rupiah pada laporan konsolidasian? | Hybrid | 6 | 0,5000 | 0,1667 | 0,5000 |
| 17 | JECX | Berapa harga penawaran saham JECX pada prospektus ini? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 17 | JECX | Berapa harga penawaran saham JECX pada prospektus ini? | Sparse | 3 | 0,6667 | 0,1667 | 0,5000 |
| 17 | JECX | Berapa harga penawaran saham JECX pada prospektus ini? | Sparse | 6 | 0,3333 | 0,1667 | 0,5000 |
| 17 | JECX | Berapa harga penawaran saham JECX pada prospektus ini? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 17 | JECX | Berapa harga penawaran saham JECX pada prospektus ini? | Dense | 3 | 0,3333 | 0,0833 | 0,5000 |
| 17 | JECX | Berapa harga penawaran saham JECX pada prospektus ini? | Dense | 6 | 0,1667 | 0,0833 | 0,5000 |
| 17 | JECX | Berapa harga penawaran saham JECX pada prospektus ini? | Hybrid | 1 | 1,0000 | 0,0833 | 1,0000 |
| 17 | JECX | Berapa harga penawaran saham JECX pada prospektus ini? | Hybrid | 3 | 0,3333 | 0,0833 | 1,0000 |
| 17 | JECX | Berapa harga penawaran saham JECX pada prospektus ini? | Hybrid | 6 | 0,3333 | 0,1667 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek IPO JECX? | Sparse | 1 | 1,0000 | 0,1250 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek IPO JECX? | Sparse | 3 | 0,3333 | 0,1250 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek IPO JECX? | Sparse | 6 | 0,5000 | 0,3750 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek IPO JECX? | Dense | 1 | 1,0000 | 0,1250 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek IPO JECX? | Dense | 3 | 1,0000 | 0,3750 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek IPO JECX? | Dense | 6 | 0,5000 | 0,3750 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek IPO JECX? | Hybrid | 1 | 1,0000 | 0,1250 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek IPO JECX? | Hybrid | 3 | 0,6667 | 0,2500 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek IPO JECX? | Hybrid | 6 | 0,5000 | 0,3750 | 1,0000 |
| 19 | JECX | Bisnis utama JECX bergerak di bidang apa? | Sparse | 1 | 1,0000 | 0,0625 | 1,0000 |
| 19 | JECX | Bisnis utama JECX bergerak di bidang apa? | Sparse | 3 | 0,6667 | 0,1250 | 1,0000 |
| 19 | JECX | Bisnis utama JECX bergerak di bidang apa? | Sparse | 6 | 0,3333 | 0,1250 | 1,0000 |
| 19 | JECX | Bisnis utama JECX bergerak di bidang apa? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 19 | JECX | Bisnis utama JECX bergerak di bidang apa? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 19 | JECX | Bisnis utama JECX bergerak di bidang apa? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 19 | JECX | Bisnis utama JECX bergerak di bidang apa? | Hybrid | 1 | 1,0000 | 0,0625 | 1,0000 |
| 19 | JECX | Bisnis utama JECX bergerak di bidang apa? | Hybrid | 3 | 0,6667 | 0,1250 | 1,0000 |
| 19 | JECX | Bisnis utama JECX bergerak di bidang apa? | Hybrid | 6 | 0,3333 | 0,1250 | 1,0000 |
| 20 | JECX | Apa ancaman bisnis yang disebut paling utama bagi JECX? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 20 | JECX | Apa ancaman bisnis yang disebut paling utama bagi JECX? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 20 | JECX | Apa ancaman bisnis yang disebut paling utama bagi JECX? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 20 | JECX | Apa ancaman bisnis yang disebut paling utama bagi JECX? | Dense | 1 | 1,0000 | 0,2500 | 1,0000 |
| 20 | JECX | Apa ancaman bisnis yang disebut paling utama bagi JECX? | Dense | 3 | 0,3333 | 0,2500 | 1,0000 |
| 20 | JECX | Apa ancaman bisnis yang disebut paling utama bagi JECX? | Dense | 6 | 0,1667 | 0,2500 | 1,0000 |
| 20 | JECX | Apa ancaman bisnis yang disebut paling utama bagi JECX? | Hybrid | 1 | 1,0000 | 0,2500 | 1,0000 |
| 20 | JECX | Apa ancaman bisnis yang disebut paling utama bagi JECX? | Hybrid | 3 | 0,3333 | 0,2500 | 1,0000 |
| 20 | JECX | Apa ancaman bisnis yang disebut paling utama bagi JECX? | Hybrid | 6 | 0,1667 | 0,2500 | 1,0000 |
| 21 | JECX | Berapa total aset konsolidasian JECX per 31 Desember 2025 dalam jutaan Rupiah? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 21 | JECX | Berapa total aset konsolidasian JECX per 31 Desember 2025 dalam jutaan Rupiah? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 21 | JECX | Berapa total aset konsolidasian JECX per 31 Desember 2025 dalam jutaan Rupiah? | Sparse | 6 | 0,1667 | 0,2000 | 0,1667 |
| 21 | JECX | Berapa total aset konsolidasian JECX per 31 Desember 2025 dalam jutaan Rupiah? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 21 | JECX | Berapa total aset konsolidasian JECX per 31 Desember 2025 dalam jutaan Rupiah? | Dense | 3 | 0,3333 | 0,2000 | 0,5000 |
| 21 | JECX | Berapa total aset konsolidasian JECX per 31 Desember 2025 dalam jutaan Rupiah? | Dense | 6 | 0,1667 | 0,2000 | 0,5000 |
| 21 | JECX | Berapa total aset konsolidasian JECX per 31 Desember 2025 dalam jutaan Rupiah? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 21 | JECX | Berapa total aset konsolidasian JECX per 31 Desember 2025 dalam jutaan Rupiah? | Hybrid | 3 | 0,3333 | 0,2000 | 0,3333 |
| 21 | JECX | Berapa total aset konsolidasian JECX per 31 Desember 2025 dalam jutaan Rupiah? | Hybrid | 6 | 0,1667 | 0,2000 | 0,3333 |
| 22 | JECX | Berapa pendapatan konsolidasian JECX untuk tahun 2025 dalam jutaan Rupiah? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 22 | JECX | Berapa pendapatan konsolidasian JECX untuk tahun 2025 dalam jutaan Rupiah? | Sparse | 3 | 0,3333 | 0,1250 | 0,3333 |
| 22 | JECX | Berapa pendapatan konsolidasian JECX untuk tahun 2025 dalam jutaan Rupiah? | Sparse | 6 | 0,5000 | 0,3750 | 0,3333 |
| 22 | JECX | Berapa pendapatan konsolidasian JECX untuk tahun 2025 dalam jutaan Rupiah? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 22 | JECX | Berapa pendapatan konsolidasian JECX untuk tahun 2025 dalam jutaan Rupiah? | Dense | 3 | 0,3333 | 0,1250 | 0,5000 |
| 22 | JECX | Berapa pendapatan konsolidasian JECX untuk tahun 2025 dalam jutaan Rupiah? | Dense | 6 | 0,3333 | 0,2500 | 0,5000 |
| 22 | JECX | Berapa pendapatan konsolidasian JECX untuk tahun 2025 dalam jutaan Rupiah? | Hybrid | 1 | 1,0000 | 0,1250 | 1,0000 |
| 22 | JECX | Berapa pendapatan konsolidasian JECX untuk tahun 2025 dalam jutaan Rupiah? | Hybrid | 3 | 0,3333 | 0,1250 | 1,0000 |
| 22 | JECX | Berapa pendapatan konsolidasian JECX untuk tahun 2025 dalam jutaan Rupiah? | Hybrid | 6 | 0,3333 | 0,2500 | 1,0000 |
| 23 | JECX | Jelaskan bidang usaha utama JECX dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 23 | JECX | Jelaskan bidang usaha utama JECX dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 23 | JECX | Jelaskan bidang usaha utama JECX dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 23 | JECX | Jelaskan bidang usaha utama JECX dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 23 | JECX | Jelaskan bidang usaha utama JECX dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 23 | JECX | Jelaskan bidang usaha utama JECX dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 23 | JECX | Jelaskan bidang usaha utama JECX dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 23 | JECX | Jelaskan bidang usaha utama JECX dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 23 | JECX | Jelaskan bidang usaha utama JECX dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 6 | 0,0000 | 0,0000 | 0,0000 |
| 24 | JECX | Apa risiko utama JECX, dan berapa pendapatan tahun 2025 dalam jutaan Rupiah pada laporan konsolidasian? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 24 | JECX | Apa risiko utama JECX, dan berapa pendapatan tahun 2025 dalam jutaan Rupiah pada laporan konsolidasian? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 24 | JECX | Apa risiko utama JECX, dan berapa pendapatan tahun 2025 dalam jutaan Rupiah pada laporan konsolidasian? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 24 | JECX | Apa risiko utama JECX, dan berapa pendapatan tahun 2025 dalam jutaan Rupiah pada laporan konsolidasian? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 24 | JECX | Apa risiko utama JECX, dan berapa pendapatan tahun 2025 dalam jutaan Rupiah pada laporan konsolidasian? | Dense | 3 | 0,3333 | 0,0833 | 0,5000 |
| 24 | JECX | Apa risiko utama JECX, dan berapa pendapatan tahun 2025 dalam jutaan Rupiah pada laporan konsolidasian? | Dense | 6 | 0,3333 | 0,1667 | 0,5000 |
| 24 | JECX | Apa risiko utama JECX, dan berapa pendapatan tahun 2025 dalam jutaan Rupiah pada laporan konsolidasian? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 24 | JECX | Apa risiko utama JECX, dan berapa pendapatan tahun 2025 dalam jutaan Rupiah pada laporan konsolidasian? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 24 | JECX | Apa risiko utama JECX, dan berapa pendapatan tahun 2025 dalam jutaan Rupiah pada laporan konsolidasian? | Hybrid | 6 | 0,1667 | 0,0833 | 0,2500 |
| 25 | JELI | Berapa harga penawaran saham JELI pada prospektus ini? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 25 | JELI | Berapa harga penawaran saham JELI pada prospektus ini? | Sparse | 3 | 0,3333 | 0,1250 | 0,5000 |
| 25 | JELI | Berapa harga penawaran saham JELI pada prospektus ini? | Sparse | 6 | 0,1667 | 0,1250 | 0,5000 |
| 25 | JELI | Berapa harga penawaran saham JELI pada prospektus ini? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 25 | JELI | Berapa harga penawaran saham JELI pada prospektus ini? | Dense | 3 | 0,6667 | 0,2500 | 0,5000 |
| 25 | JELI | Berapa harga penawaran saham JELI pada prospektus ini? | Dense | 6 | 0,3333 | 0,2500 | 0,5000 |
| 25 | JELI | Berapa harga penawaran saham JELI pada prospektus ini? | Hybrid | 1 | 1,0000 | 0,1250 | 1,0000 |
| 25 | JELI | Berapa harga penawaran saham JELI pada prospektus ini? | Hybrid | 3 | 0,6667 | 0,2500 | 1,0000 |
| 25 | JELI | Berapa harga penawaran saham JELI pada prospektus ini? | Hybrid | 6 | 0,3333 | 0,2500 | 1,0000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek IPO JELI? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek IPO JELI? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek IPO JELI? | Sparse | 6 | 0,3333 | 0,3333 | 0,2000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek IPO JELI? | Dense | 1 | 1,0000 | 0,1667 | 1,0000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek IPO JELI? | Dense | 3 | 0,6667 | 0,3333 | 1,0000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek IPO JELI? | Dense | 6 | 0,3333 | 0,3333 | 1,0000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek IPO JELI? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek IPO JELI? | Hybrid | 3 | 0,6667 | 0,3333 | 0,5000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek IPO JELI? | Hybrid | 6 | 0,5000 | 0,5000 | 0,5000 |
| 27 | JELI | Bisnis utama JELI bergerak di bidang apa? | Sparse | 1 | 1,0000 | 0,1111 | 1,0000 |
| 27 | JELI | Bisnis utama JELI bergerak di bidang apa? | Sparse | 3 | 0,3333 | 0,1111 | 1,0000 |
| 27 | JELI | Bisnis utama JELI bergerak di bidang apa? | Sparse | 6 | 0,1667 | 0,1111 | 1,0000 |
| 27 | JELI | Bisnis utama JELI bergerak di bidang apa? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 27 | JELI | Bisnis utama JELI bergerak di bidang apa? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 27 | JELI | Bisnis utama JELI bergerak di bidang apa? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 27 | JELI | Bisnis utama JELI bergerak di bidang apa? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 27 | JELI | Bisnis utama JELI bergerak di bidang apa? | Hybrid | 3 | 0,3333 | 0,1111 | 0,3333 |
| 27 | JELI | Bisnis utama JELI bergerak di bidang apa? | Hybrid | 6 | 0,1667 | 0,1111 | 0,3333 |
| 28 | JELI | Apa ancaman bisnis yang disebut paling utama bagi JELI? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 28 | JELI | Apa ancaman bisnis yang disebut paling utama bagi JELI? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 28 | JELI | Apa ancaman bisnis yang disebut paling utama bagi JELI? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 28 | JELI | Apa ancaman bisnis yang disebut paling utama bagi JELI? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 28 | JELI | Apa ancaman bisnis yang disebut paling utama bagi JELI? | Dense | 3 | 0,3333 | 0,1429 | 0,5000 |
| 28 | JELI | Apa ancaman bisnis yang disebut paling utama bagi JELI? | Dense | 6 | 0,1667 | 0,1429 | 0,5000 |
| 28 | JELI | Apa ancaman bisnis yang disebut paling utama bagi JELI? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 28 | JELI | Apa ancaman bisnis yang disebut paling utama bagi JELI? | Hybrid | 3 | 0,3333 | 0,1429 | 0,3333 |
| 28 | JELI | Apa ancaman bisnis yang disebut paling utama bagi JELI? | Hybrid | 6 | 0,1667 | 0,1429 | 0,3333 |
| 29 | JELI | Berapa total aset konsolidasian JELI per 31 Desember 2025 dalam Rupiah? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 29 | JELI | Berapa total aset konsolidasian JELI per 31 Desember 2025 dalam Rupiah? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 29 | JELI | Berapa total aset konsolidasian JELI per 31 Desember 2025 dalam Rupiah? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 29 | JELI | Berapa total aset konsolidasian JELI per 31 Desember 2025 dalam Rupiah? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 29 | JELI | Berapa total aset konsolidasian JELI per 31 Desember 2025 dalam Rupiah? | Dense | 3 | 0,3333 | 0,2000 | 0,5000 |
| 29 | JELI | Berapa total aset konsolidasian JELI per 31 Desember 2025 dalam Rupiah? | Dense | 6 | 0,3333 | 0,4000 | 0,5000 |
| 29 | JELI | Berapa total aset konsolidasian JELI per 31 Desember 2025 dalam Rupiah? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 29 | JELI | Berapa total aset konsolidasian JELI per 31 Desember 2025 dalam Rupiah? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 29 | JELI | Berapa total aset konsolidasian JELI per 31 Desember 2025 dalam Rupiah? | Hybrid | 6 | 0,1667 | 0,2000 | 0,2000 |
| 30 | JELI | Berapa penjualan konsolidasian JELI untuk tahun 2025 dalam Rupiah? | Sparse | 1 | 1,0000 | 0,1429 | 1,0000 |
| 30 | JELI | Berapa penjualan konsolidasian JELI untuk tahun 2025 dalam Rupiah? | Sparse | 3 | 0,6667 | 0,2857 | 1,0000 |
| 30 | JELI | Berapa penjualan konsolidasian JELI untuk tahun 2025 dalam Rupiah? | Sparse | 6 | 0,3333 | 0,2857 | 1,0000 |
| 30 | JELI | Berapa penjualan konsolidasian JELI untuk tahun 2025 dalam Rupiah? | Dense | 1 | 1,0000 | 0,1429 | 1,0000 |
| 30 | JELI | Berapa penjualan konsolidasian JELI untuk tahun 2025 dalam Rupiah? | Dense | 3 | 0,6667 | 0,2857 | 1,0000 |
| 30 | JELI | Berapa penjualan konsolidasian JELI untuk tahun 2025 dalam Rupiah? | Dense | 6 | 0,5000 | 0,4286 | 1,0000 |
| 30 | JELI | Berapa penjualan konsolidasian JELI untuk tahun 2025 dalam Rupiah? | Hybrid | 1 | 1,0000 | 0,1429 | 1,0000 |
| 30 | JELI | Berapa penjualan konsolidasian JELI untuk tahun 2025 dalam Rupiah? | Hybrid | 3 | 0,6667 | 0,2857 | 1,0000 |
| 30 | JELI | Berapa penjualan konsolidasian JELI untuk tahun 2025 dalam Rupiah? | Hybrid | 6 | 0,5000 | 0,4286 | 1,0000 |
| 31 | JELI | Jelaskan bidang usaha utama JELI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 31 | JELI | Jelaskan bidang usaha utama JELI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 31 | JELI | Jelaskan bidang usaha utama JELI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 31 | JELI | Jelaskan bidang usaha utama JELI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 31 | JELI | Jelaskan bidang usaha utama JELI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 3 | 0,3333 | 0,0833 | 0,3333 |
| 31 | JELI | Jelaskan bidang usaha utama JELI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 6 | 0,1667 | 0,0833 | 0,3333 |
| 31 | JELI | Jelaskan bidang usaha utama JELI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 31 | JELI | Jelaskan bidang usaha utama JELI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 3 | 0,3333 | 0,0833 | 0,3333 |
| 31 | JELI | Jelaskan bidang usaha utama JELI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 6 | 0,1667 | 0,0833 | 0,3333 |
| 32 | JELI | Apa risiko utama JELI, dan berapa penjualan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 32 | JELI | Apa risiko utama JELI, dan berapa penjualan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 32 | JELI | Apa risiko utama JELI, dan berapa penjualan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 32 | JELI | Apa risiko utama JELI, dan berapa penjualan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 32 | JELI | Apa risiko utama JELI, dan berapa penjualan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Dense | 3 | 0,3333 | 0,0714 | 0,5000 |
| 32 | JELI | Apa risiko utama JELI, dan berapa penjualan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Dense | 6 | 0,3333 | 0,1429 | 0,5000 |
| 32 | JELI | Apa risiko utama JELI, dan berapa penjualan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Hybrid | 1 | 1,0000 | 0,0714 | 1,0000 |
| 32 | JELI | Apa risiko utama JELI, dan berapa penjualan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Hybrid | 3 | 0,3333 | 0,0714 | 1,0000 |
| 32 | JELI | Apa risiko utama JELI, dan berapa penjualan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Hybrid | 6 | 0,1667 | 0,0714 | 1,0000 |
| 33 | PRDL | Berapa harga penawaran saham PRDL pada prospektus ini? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 33 | PRDL | Berapa harga penawaran saham PRDL pada prospektus ini? | Sparse | 3 | 0,3333 | 0,1111 | 0,5000 |
| 33 | PRDL | Berapa harga penawaran saham PRDL pada prospektus ini? | Sparse | 6 | 0,1667 | 0,1111 | 0,5000 |
| 33 | PRDL | Berapa harga penawaran saham PRDL pada prospektus ini? | Dense | 1 | 1,0000 | 0,1111 | 1,0000 |
| 33 | PRDL | Berapa harga penawaran saham PRDL pada prospektus ini? | Dense | 3 | 0,3333 | 0,1111 | 1,0000 |
| 33 | PRDL | Berapa harga penawaran saham PRDL pada prospektus ini? | Dense | 6 | 0,3333 | 0,2222 | 1,0000 |
| 33 | PRDL | Berapa harga penawaran saham PRDL pada prospektus ini? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 33 | PRDL | Berapa harga penawaran saham PRDL pada prospektus ini? | Hybrid | 3 | 0,3333 | 0,1111 | 0,5000 |
| 33 | PRDL | Berapa harga penawaran saham PRDL pada prospektus ini? | Hybrid | 6 | 0,1667 | 0,1111 | 0,5000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek IPO PRDL? | Sparse | 1 | 1,0000 | 0,1250 | 1,0000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek IPO PRDL? | Sparse | 3 | 0,3333 | 0,1250 | 1,0000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek IPO PRDL? | Sparse | 6 | 0,1667 | 0,1250 | 1,0000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek IPO PRDL? | Dense | 1 | 1,0000 | 0,1250 | 1,0000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek IPO PRDL? | Dense | 3 | 0,6667 | 0,2500 | 1,0000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek IPO PRDL? | Dense | 6 | 0,5000 | 0,3750 | 1,0000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek IPO PRDL? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek IPO PRDL? | Hybrid | 3 | 0,3333 | 0,1250 | 0,3333 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek IPO PRDL? | Hybrid | 6 | 0,6667 | 0,5000 | 0,3333 |
| 35 | PRDL | Bisnis utama PRDL bergerak di bidang apa? | Sparse | 1 | 1,0000 | 0,1429 | 1,0000 |
| 35 | PRDL | Bisnis utama PRDL bergerak di bidang apa? | Sparse | 3 | 1,0000 | 0,4286 | 1,0000 |
| 35 | PRDL | Bisnis utama PRDL bergerak di bidang apa? | Sparse | 6 | 0,5000 | 0,4286 | 1,0000 |
| 35 | PRDL | Bisnis utama PRDL bergerak di bidang apa? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 35 | PRDL | Bisnis utama PRDL bergerak di bidang apa? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 35 | PRDL | Bisnis utama PRDL bergerak di bidang apa? | Dense | 6 | 0,1667 | 0,1429 | 0,2500 |
| 35 | PRDL | Bisnis utama PRDL bergerak di bidang apa? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 35 | PRDL | Bisnis utama PRDL bergerak di bidang apa? | Hybrid | 3 | 0,3333 | 0,1429 | 0,5000 |
| 35 | PRDL | Bisnis utama PRDL bergerak di bidang apa? | Hybrid | 6 | 0,5000 | 0,4286 | 0,5000 |
| 36 | PRDL | Apa ancaman bisnis yang disebut paling utama bagi PRDL? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 36 | PRDL | Apa ancaman bisnis yang disebut paling utama bagi PRDL? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 36 | PRDL | Apa ancaman bisnis yang disebut paling utama bagi PRDL? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 36 | PRDL | Apa ancaman bisnis yang disebut paling utama bagi PRDL? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 36 | PRDL | Apa ancaman bisnis yang disebut paling utama bagi PRDL? | Dense | 3 | 0,3333 | 0,1667 | 0,5000 |
| 36 | PRDL | Apa ancaman bisnis yang disebut paling utama bagi PRDL? | Dense | 6 | 0,1667 | 0,1667 | 0,5000 |
| 36 | PRDL | Apa ancaman bisnis yang disebut paling utama bagi PRDL? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 36 | PRDL | Apa ancaman bisnis yang disebut paling utama bagi PRDL? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 36 | PRDL | Apa ancaman bisnis yang disebut paling utama bagi PRDL? | Hybrid | 6 | 0,1667 | 0,1667 | 0,2500 |
| 37 | PRDL | Berapa total aset Perseroan PRDL per 31 Desember 2025 dalam ribuan Rupiah? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 37 | PRDL | Berapa total aset Perseroan PRDL per 31 Desember 2025 dalam ribuan Rupiah? | Sparse | 3 | 0,3333 | 0,1667 | 0,5000 |
| 37 | PRDL | Berapa total aset Perseroan PRDL per 31 Desember 2025 dalam ribuan Rupiah? | Sparse | 6 | 0,1667 | 0,1667 | 0,5000 |
| 37 | PRDL | Berapa total aset Perseroan PRDL per 31 Desember 2025 dalam ribuan Rupiah? | Dense | 1 | 1,0000 | 0,1667 | 1,0000 |
| 37 | PRDL | Berapa total aset Perseroan PRDL per 31 Desember 2025 dalam ribuan Rupiah? | Dense | 3 | 0,3333 | 0,1667 | 1,0000 |
| 37 | PRDL | Berapa total aset Perseroan PRDL per 31 Desember 2025 dalam ribuan Rupiah? | Dense | 6 | 0,3333 | 0,3333 | 1,0000 |
| 37 | PRDL | Berapa total aset Perseroan PRDL per 31 Desember 2025 dalam ribuan Rupiah? | Hybrid | 1 | 1,0000 | 0,1667 | 1,0000 |
| 37 | PRDL | Berapa total aset Perseroan PRDL per 31 Desember 2025 dalam ribuan Rupiah? | Hybrid | 3 | 0,3333 | 0,1667 | 1,0000 |
| 37 | PRDL | Berapa total aset Perseroan PRDL per 31 Desember 2025 dalam ribuan Rupiah? | Hybrid | 6 | 0,1667 | 0,1667 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun 2025 dalam ribuan Rupiah? | Sparse | 1 | 1,0000 | 0,1111 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun 2025 dalam ribuan Rupiah? | Sparse | 3 | 0,6667 | 0,2222 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun 2025 dalam ribuan Rupiah? | Sparse | 6 | 0,3333 | 0,2222 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun 2025 dalam ribuan Rupiah? | Dense | 1 | 1,0000 | 0,1111 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun 2025 dalam ribuan Rupiah? | Dense | 3 | 0,6667 | 0,2222 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun 2025 dalam ribuan Rupiah? | Dense | 6 | 0,6667 | 0,4444 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun 2025 dalam ribuan Rupiah? | Hybrid | 1 | 1,0000 | 0,1111 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun 2025 dalam ribuan Rupiah? | Hybrid | 3 | 0,6667 | 0,2222 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun 2025 dalam ribuan Rupiah? | Hybrid | 6 | 0,5000 | 0,3333 | 1,0000 |
| 39 | PRDL | Jelaskan bidang usaha utama PRDL dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 39 | PRDL | Jelaskan bidang usaha utama PRDL dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 39 | PRDL | Jelaskan bidang usaha utama PRDL dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 39 | PRDL | Jelaskan bidang usaha utama PRDL dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 39 | PRDL | Jelaskan bidang usaha utama PRDL dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 39 | PRDL | Jelaskan bidang usaha utama PRDL dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 39 | PRDL | Jelaskan bidang usaha utama PRDL dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 39 | PRDL | Jelaskan bidang usaha utama PRDL dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 39 | PRDL | Jelaskan bidang usaha utama PRDL dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 6 | 0,0000 | 0,0000 | 0,0000 |
| 40 | PRDL | Apa risiko utama PRDL, dan berapa pendapatan dari kontrak dengan pelanggan tahun 2025 dalam ribuan Rupiah pada laporan Perseroan? | Sparse | 1 | 1,0000 | 0,0667 | 1,0000 |
| 40 | PRDL | Apa risiko utama PRDL, dan berapa pendapatan dari kontrak dengan pelanggan tahun 2025 dalam ribuan Rupiah pada laporan Perseroan? | Sparse | 3 | 0,6667 | 0,1333 | 1,0000 |
| 40 | PRDL | Apa risiko utama PRDL, dan berapa pendapatan dari kontrak dengan pelanggan tahun 2025 dalam ribuan Rupiah pada laporan Perseroan? | Sparse | 6 | 0,5000 | 0,2000 | 1,0000 |
| 40 | PRDL | Apa risiko utama PRDL, dan berapa pendapatan dari kontrak dengan pelanggan tahun 2025 dalam ribuan Rupiah pada laporan Perseroan? | Dense | 1 | 1,0000 | 0,0667 | 1,0000 |
| 40 | PRDL | Apa risiko utama PRDL, dan berapa pendapatan dari kontrak dengan pelanggan tahun 2025 dalam ribuan Rupiah pada laporan Perseroan? | Dense | 3 | 0,3333 | 0,0667 | 1,0000 |
| 40 | PRDL | Apa risiko utama PRDL, dan berapa pendapatan dari kontrak dengan pelanggan tahun 2025 dalam ribuan Rupiah pada laporan Perseroan? | Dense | 6 | 0,6667 | 0,2667 | 1,0000 |
| 40 | PRDL | Apa risiko utama PRDL, dan berapa pendapatan dari kontrak dengan pelanggan tahun 2025 dalam ribuan Rupiah pada laporan Perseroan? | Hybrid | 1 | 1,0000 | 0,0667 | 1,0000 |
| 40 | PRDL | Apa risiko utama PRDL, dan berapa pendapatan dari kontrak dengan pelanggan tahun 2025 dalam ribuan Rupiah pada laporan Perseroan? | Hybrid | 3 | 0,3333 | 0,0667 | 1,0000 |
| 40 | PRDL | Apa risiko utama PRDL, dan berapa pendapatan dari kontrak dengan pelanggan tahun 2025 dalam ribuan Rupiah pada laporan Perseroan? | Hybrid | 6 | 0,6667 | 0,2667 | 1,0000 |
| 41 | RANS | Berapa harga penawaran saham RANS pada prospektus ini? | Sparse | 1 | 1,0000 | 0,1111 | 1,0000 |
| 41 | RANS | Berapa harga penawaran saham RANS pada prospektus ini? | Sparse | 3 | 0,6667 | 0,2222 | 1,0000 |
| 41 | RANS | Berapa harga penawaran saham RANS pada prospektus ini? | Sparse | 6 | 0,3333 | 0,2222 | 1,0000 |
| 41 | RANS | Berapa harga penawaran saham RANS pada prospektus ini? | Dense | 1 | 1,0000 | 0,1111 | 1,0000 |
| 41 | RANS | Berapa harga penawaran saham RANS pada prospektus ini? | Dense | 3 | 0,3333 | 0,1111 | 1,0000 |
| 41 | RANS | Berapa harga penawaran saham RANS pada prospektus ini? | Dense | 6 | 0,1667 | 0,1111 | 1,0000 |
| 41 | RANS | Berapa harga penawaran saham RANS pada prospektus ini? | Hybrid | 1 | 1,0000 | 0,1111 | 1,0000 |
| 41 | RANS | Berapa harga penawaran saham RANS pada prospektus ini? | Hybrid | 3 | 0,6667 | 0,2222 | 1,0000 |
| 41 | RANS | Berapa harga penawaran saham RANS pada prospektus ini? | Hybrid | 6 | 0,5000 | 0,3333 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek IPO RANS? | Sparse | 1 | 1,0000 | 0,1250 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek IPO RANS? | Sparse | 3 | 0,3333 | 0,1250 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek IPO RANS? | Sparse | 6 | 0,3333 | 0,2500 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek IPO RANS? | Dense | 1 | 1,0000 | 0,1250 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek IPO RANS? | Dense | 3 | 0,6667 | 0,2500 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek IPO RANS? | Dense | 6 | 0,3333 | 0,2500 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek IPO RANS? | Hybrid | 1 | 1,0000 | 0,1250 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek IPO RANS? | Hybrid | 3 | 0,6667 | 0,2500 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek IPO RANS? | Hybrid | 6 | 0,5000 | 0,3750 | 1,0000 |
| 43 | RANS | Bisnis utama RANS bergerak di bidang apa? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 43 | RANS | Bisnis utama RANS bergerak di bidang apa? | Sparse | 3 | 0,3333 | 0,0385 | 0,3333 |
| 43 | RANS | Bisnis utama RANS bergerak di bidang apa? | Sparse | 6 | 0,3333 | 0,0769 | 0,3333 |
| 43 | RANS | Bisnis utama RANS bergerak di bidang apa? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 43 | RANS | Bisnis utama RANS bergerak di bidang apa? | Dense | 3 | 0,3333 | 0,0385 | 0,5000 |
| 43 | RANS | Bisnis utama RANS bergerak di bidang apa? | Dense | 6 | 0,1667 | 0,0385 | 0,5000 |
| 43 | RANS | Bisnis utama RANS bergerak di bidang apa? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 43 | RANS | Bisnis utama RANS bergerak di bidang apa? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 43 | RANS | Bisnis utama RANS bergerak di bidang apa? | Hybrid | 6 | 0,3333 | 0,0769 | 0,2500 |
| 44 | RANS | Apa ancaman bisnis yang disebut paling utama bagi RANS? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 44 | RANS | Apa ancaman bisnis yang disebut paling utama bagi RANS? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 44 | RANS | Apa ancaman bisnis yang disebut paling utama bagi RANS? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 44 | RANS | Apa ancaman bisnis yang disebut paling utama bagi RANS? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 44 | RANS | Apa ancaman bisnis yang disebut paling utama bagi RANS? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 44 | RANS | Apa ancaman bisnis yang disebut paling utama bagi RANS? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 44 | RANS | Apa ancaman bisnis yang disebut paling utama bagi RANS? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 44 | RANS | Apa ancaman bisnis yang disebut paling utama bagi RANS? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 44 | RANS | Apa ancaman bisnis yang disebut paling utama bagi RANS? | Hybrid | 6 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Berapa total aset konsolidasian RANS per 31 Desember 2025 dalam Rupiah? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Berapa total aset konsolidasian RANS per 31 Desember 2025 dalam Rupiah? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Berapa total aset konsolidasian RANS per 31 Desember 2025 dalam Rupiah? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Berapa total aset konsolidasian RANS per 31 Desember 2025 dalam Rupiah? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Berapa total aset konsolidasian RANS per 31 Desember 2025 dalam Rupiah? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Berapa total aset konsolidasian RANS per 31 Desember 2025 dalam Rupiah? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Berapa total aset konsolidasian RANS per 31 Desember 2025 dalam Rupiah? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Berapa total aset konsolidasian RANS per 31 Desember 2025 dalam Rupiah? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Berapa total aset konsolidasian RANS per 31 Desember 2025 dalam Rupiah? | Hybrid | 6 | 0,0000 | 0,0000 | 0,0000 |
| 46 | RANS | Berapa pendapatan konsolidasian RANS untuk tahun 2025 dalam Rupiah? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 46 | RANS | Berapa pendapatan konsolidasian RANS untuk tahun 2025 dalam Rupiah? | Sparse | 3 | 0,3333 | 0,1111 | 0,3333 |
| 46 | RANS | Berapa pendapatan konsolidasian RANS untuk tahun 2025 dalam Rupiah? | Sparse | 6 | 0,1667 | 0,1111 | 0,3333 |
| 46 | RANS | Berapa pendapatan konsolidasian RANS untuk tahun 2025 dalam Rupiah? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 46 | RANS | Berapa pendapatan konsolidasian RANS untuk tahun 2025 dalam Rupiah? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 46 | RANS | Berapa pendapatan konsolidasian RANS untuk tahun 2025 dalam Rupiah? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 46 | RANS | Berapa pendapatan konsolidasian RANS untuk tahun 2025 dalam Rupiah? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 46 | RANS | Berapa pendapatan konsolidasian RANS untuk tahun 2025 dalam Rupiah? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 46 | RANS | Berapa pendapatan konsolidasian RANS untuk tahun 2025 dalam Rupiah? | Hybrid | 6 | 0,1667 | 0,1111 | 0,1667 |
| 47 | RANS | Jelaskan bidang usaha utama RANS dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Jelaskan bidang usaha utama RANS dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Jelaskan bidang usaha utama RANS dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Jelaskan bidang usaha utama RANS dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Jelaskan bidang usaha utama RANS dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Jelaskan bidang usaha utama RANS dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Jelaskan bidang usaha utama RANS dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Jelaskan bidang usaha utama RANS dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Jelaskan bidang usaha utama RANS dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 6 | 0,0000 | 0,0000 | 0,0000 |
| 48 | RANS | Apa risiko utama RANS, dan berapa pendapatan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Sparse | 1 | 1,0000 | 0,0667 | 1,0000 |
| 48 | RANS | Apa risiko utama RANS, dan berapa pendapatan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Sparse | 3 | 0,3333 | 0,0667 | 1,0000 |
| 48 | RANS | Apa risiko utama RANS, dan berapa pendapatan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Sparse | 6 | 0,1667 | 0,0667 | 1,0000 |
| 48 | RANS | Apa risiko utama RANS, dan berapa pendapatan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 48 | RANS | Apa risiko utama RANS, dan berapa pendapatan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 48 | RANS | Apa risiko utama RANS, dan berapa pendapatan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 48 | RANS | Apa risiko utama RANS, dan berapa pendapatan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 48 | RANS | Apa risiko utama RANS, dan berapa pendapatan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Hybrid | 3 | 0,3333 | 0,0667 | 0,5000 |
| 48 | RANS | Apa risiko utama RANS, dan berapa pendapatan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Hybrid | 6 | 0,1667 | 0,0667 | 0,5000 |
| 49 | SWAP | Berapa harga penawaran saham SWAP pada prospektus ini? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 49 | SWAP | Berapa harga penawaran saham SWAP pada prospektus ini? | Sparse | 3 | 0,3333 | 0,2500 | 0,5000 |
| 49 | SWAP | Berapa harga penawaran saham SWAP pada prospektus ini? | Sparse | 6 | 0,1667 | 0,2500 | 0,5000 |
| 49 | SWAP | Berapa harga penawaran saham SWAP pada prospektus ini? | Dense | 1 | 1,0000 | 0,2500 | 1,0000 |
| 49 | SWAP | Berapa harga penawaran saham SWAP pada prospektus ini? | Dense | 3 | 0,3333 | 0,2500 | 1,0000 |
| 49 | SWAP | Berapa harga penawaran saham SWAP pada prospektus ini? | Dense | 6 | 0,1667 | 0,2500 | 1,0000 |
| 49 | SWAP | Berapa harga penawaran saham SWAP pada prospektus ini? | Hybrid | 1 | 1,0000 | 0,2500 | 1,0000 |
| 49 | SWAP | Berapa harga penawaran saham SWAP pada prospektus ini? | Hybrid | 3 | 0,6667 | 0,5000 | 1,0000 |
| 49 | SWAP | Berapa harga penawaran saham SWAP pada prospektus ini? | Hybrid | 6 | 0,3333 | 0,5000 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek IPO SWAP? | Sparse | 1 | 1,0000 | 0,1429 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek IPO SWAP? | Sparse | 3 | 1,0000 | 0,4286 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek IPO SWAP? | Sparse | 6 | 0,6667 | 0,5714 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek IPO SWAP? | Dense | 1 | 1,0000 | 0,1429 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek IPO SWAP? | Dense | 3 | 1,0000 | 0,4286 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek IPO SWAP? | Dense | 6 | 0,6667 | 0,5714 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek IPO SWAP? | Hybrid | 1 | 1,0000 | 0,1429 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek IPO SWAP? | Hybrid | 3 | 1,0000 | 0,4286 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek IPO SWAP? | Hybrid | 6 | 0,8333 | 0,7143 | 1,0000 |
| 51 | SWAP | Bisnis utama SWAP bergerak di bidang apa? | Sparse | 1 | 1,0000 | 0,1429 | 1,0000 |
| 51 | SWAP | Bisnis utama SWAP bergerak di bidang apa? | Sparse | 3 | 0,6667 | 0,2857 | 1,0000 |
| 51 | SWAP | Bisnis utama SWAP bergerak di bidang apa? | Sparse | 6 | 0,3333 | 0,2857 | 1,0000 |
| 51 | SWAP | Bisnis utama SWAP bergerak di bidang apa? | Dense | 1 | 1,0000 | 0,1429 | 1,0000 |
| 51 | SWAP | Bisnis utama SWAP bergerak di bidang apa? | Dense | 3 | 0,6667 | 0,2857 | 1,0000 |
| 51 | SWAP | Bisnis utama SWAP bergerak di bidang apa? | Dense | 6 | 0,5000 | 0,4286 | 1,0000 |
| 51 | SWAP | Bisnis utama SWAP bergerak di bidang apa? | Hybrid | 1 | 1,0000 | 0,1429 | 1,0000 |
| 51 | SWAP | Bisnis utama SWAP bergerak di bidang apa? | Hybrid | 3 | 0,6667 | 0,2857 | 1,0000 |
| 51 | SWAP | Bisnis utama SWAP bergerak di bidang apa? | Hybrid | 6 | 0,3333 | 0,2857 | 1,0000 |
| 52 | SWAP | Apa ancaman bisnis yang disebut paling utama bagi SWAP? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 52 | SWAP | Apa ancaman bisnis yang disebut paling utama bagi SWAP? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 52 | SWAP | Apa ancaman bisnis yang disebut paling utama bagi SWAP? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 52 | SWAP | Apa ancaman bisnis yang disebut paling utama bagi SWAP? | Dense | 1 | 1,0000 | 0,2500 | 1,0000 |
| 52 | SWAP | Apa ancaman bisnis yang disebut paling utama bagi SWAP? | Dense | 3 | 0,3333 | 0,2500 | 1,0000 |
| 52 | SWAP | Apa ancaman bisnis yang disebut paling utama bagi SWAP? | Dense | 6 | 0,1667 | 0,2500 | 1,0000 |
| 52 | SWAP | Apa ancaman bisnis yang disebut paling utama bagi SWAP? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 52 | SWAP | Apa ancaman bisnis yang disebut paling utama bagi SWAP? | Hybrid | 3 | 0,3333 | 0,2500 | 0,5000 |
| 52 | SWAP | Apa ancaman bisnis yang disebut paling utama bagi SWAP? | Hybrid | 6 | 0,1667 | 0,2500 | 0,5000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP per 31 Desember 2025 dalam Rupiah? | Sparse | 1 | 1,0000 | 0,2500 | 1,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP per 31 Desember 2025 dalam Rupiah? | Sparse | 3 | 0,6667 | 0,5000 | 1,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP per 31 Desember 2025 dalam Rupiah? | Sparse | 6 | 0,5000 | 0,7500 | 1,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP per 31 Desember 2025 dalam Rupiah? | Dense | 1 | 1,0000 | 0,2500 | 1,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP per 31 Desember 2025 dalam Rupiah? | Dense | 3 | 0,3333 | 0,2500 | 1,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP per 31 Desember 2025 dalam Rupiah? | Dense | 6 | 0,3333 | 0,5000 | 1,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP per 31 Desember 2025 dalam Rupiah? | Hybrid | 1 | 1,0000 | 0,2500 | 1,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP per 31 Desember 2025 dalam Rupiah? | Hybrid | 3 | 0,6667 | 0,5000 | 1,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP per 31 Desember 2025 dalam Rupiah? | Hybrid | 6 | 0,3333 | 0,5000 | 1,0000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun 2025 dalam Rupiah? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun 2025 dalam Rupiah? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun 2025 dalam Rupiah? | Sparse | 6 | 0,1667 | 0,2000 | 0,2000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun 2025 dalam Rupiah? | Dense | 1 | 1,0000 | 0,2000 | 1,0000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun 2025 dalam Rupiah? | Dense | 3 | 0,6667 | 0,4000 | 1,0000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun 2025 dalam Rupiah? | Dense | 6 | 0,5000 | 0,6000 | 1,0000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun 2025 dalam Rupiah? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun 2025 dalam Rupiah? | Hybrid | 3 | 0,3333 | 0,2000 | 0,5000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun 2025 dalam Rupiah? | Hybrid | 6 | 0,3333 | 0,4000 | 0,5000 |
| 55 | SWAP | Jelaskan bidang usaha utama SWAP dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 55 | SWAP | Jelaskan bidang usaha utama SWAP dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 55 | SWAP | Jelaskan bidang usaha utama SWAP dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 55 | SWAP | Jelaskan bidang usaha utama SWAP dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 55 | SWAP | Jelaskan bidang usaha utama SWAP dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 55 | SWAP | Jelaskan bidang usaha utama SWAP dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 55 | SWAP | Jelaskan bidang usaha utama SWAP dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 55 | SWAP | Jelaskan bidang usaha utama SWAP dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 55 | SWAP | Jelaskan bidang usaha utama SWAP dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Hybrid | 6 | 0,0000 | 0,0000 | 0,0000 |
| 56 | SWAP | Apa risiko utama SWAP, dan berapa pendapatan usaha tahun 2025 dalam Rupiah pada laporan Perseroan? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 56 | SWAP | Apa risiko utama SWAP, dan berapa pendapatan usaha tahun 2025 dalam Rupiah pada laporan Perseroan? | Sparse | 3 | 0,3333 | 0,1111 | 0,3333 |
| 56 | SWAP | Apa risiko utama SWAP, dan berapa pendapatan usaha tahun 2025 dalam Rupiah pada laporan Perseroan? | Sparse | 6 | 0,3333 | 0,2222 | 0,3333 |
| 56 | SWAP | Apa risiko utama SWAP, dan berapa pendapatan usaha tahun 2025 dalam Rupiah pada laporan Perseroan? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 56 | SWAP | Apa risiko utama SWAP, dan berapa pendapatan usaha tahun 2025 dalam Rupiah pada laporan Perseroan? | Dense | 3 | 0,3333 | 0,1111 | 0,5000 |
| 56 | SWAP | Apa risiko utama SWAP, dan berapa pendapatan usaha tahun 2025 dalam Rupiah pada laporan Perseroan? | Dense | 6 | 0,3333 | 0,2222 | 0,5000 |
| 56 | SWAP | Apa risiko utama SWAP, dan berapa pendapatan usaha tahun 2025 dalam Rupiah pada laporan Perseroan? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 56 | SWAP | Apa risiko utama SWAP, dan berapa pendapatan usaha tahun 2025 dalam Rupiah pada laporan Perseroan? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 56 | SWAP | Apa risiko utama SWAP, dan berapa pendapatan usaha tahun 2025 dalam Rupiah pada laporan Perseroan? | Hybrid | 6 | 0,3333 | 0,2222 | 0,2500 |

## Hasil Pengujian RAGAS

In [7]:
ragas_directory = PROJECT / "evaluation/results/ragas_56_konteks"
if ULANG_RAGAS:
    ragas_directory = HASIL / ("ragas_" + RAGAS_RUN_ID)

arguments = [
    "-m", "evaluation.run_ragas_complete",
    "--questions", DATASET / "questions.jsonl",
    "--ground-truth", DATASET / "ground_truth.jsonl",
    "--output-dir", ragas_directory,
    "--context-policy", "retrieved_top_k",
    "--evaluator-max-tokens", "8192",
]
if (ragas_directory / "checkpoint.json").is_file():
    arguments.append("--resume")
jalankan(*arguments)
ragas = read_json(ragas_directory / "checkpoint.json")
ringkasan_ragas, rincian_ragas, status_ragas = ragas_tables(questions, references, ragas, retrieval)
konteks_ragas = ragas_context_table(questions, ragas)
jumlah_tanpa_konteks = status_ragas.get("skipped_no_context", 0)
cakupan_ragas = 100 * status_ragas.get("scored", 0) / len(rincian_ragas)
cakupan_teks = f"{cakupan_ragas:.2f}".replace(".", ",")
protokol_teks = (
    "RAGAS menguji retrieval dan generasi dari enam konteks teratas Hybrid, "
    "tanpa ambang penolakan aplikasi."
)
print(
    f"{len(rincian_ragas)} pertanyaan diuji: {status_ragas.get('scored', 0)} dinilai. "
    f"Cakupan penilaian: {cakupan_teks}%."
)
display(Markdown(protokol_teks))
display(Markdown(markdown_table(ringkasan_ragas)))


56 pertanyaan diuji: 56 dinilai. Cakupan penilaian: 100,00%.


RAGAS menguji retrieval dan generasi dari enam konteks teratas Hybrid, tanpa ambang penolakan aplikasi.

| Metrik | Rata-rata | Pertanyaan dinilai |
| --- | --- | --- |
| Faithfulness | 0,7307 | 56 |
| Answer Relevancy | 0,6645 | 56 |
| Context Precision | 0,4965 | 56 |
| Context Recall | 0,8006 | 56 |

### Hasil Pengujian RAGAS per Pertanyaan

In [8]:
kolom_skor_ragas = [
    "No", "Emiten", "Pertanyaan", "Faithfulness", "Answer Relevancy", "Context Precision", "Context Recall", "Status",
]
kolom_jawaban_ragas = ["No", "Emiten", "Pertanyaan", "Jawaban sistem", "Jawaban acuan", "Keterangan"]
tabel_skor_ragas = [{kolom: row[kolom] for kolom in kolom_skor_ragas} for row in rincian_ragas]
tabel_jawaban_ragas = [{kolom: row[kolom] for kolom in kolom_jawaban_ragas} for row in rincian_ragas]
display(Markdown("#### Skor RAGAS per Pertanyaan"))
display(Markdown(markdown_table(tabel_skor_ragas)))
display(Markdown("#### Jawaban Sistem dan Jawaban Acuan"))
display(Markdown(markdown_table(tabel_jawaban_ragas)))


#### Skor RAGAS per Pertanyaan

| No | Emiten | Pertanyaan | Faithfulness | Answer Relevancy | Context Precision | Context Recall | Status |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | BACH | Berapa harga penawaran saham BACH pada prospektus ini? | 1,0000 | 0,9365 | 0,5000 | 1,0000 | Dinilai |
| 2 | BACH | Siapa penjamin pelaksana emisi efek IPO BACH? | 1,0000 | 0,9315 | 0,3250 | 1,0000 | Dinilai |
| 3 | BACH | Bisnis utama BACH bergerak di bidang apa? | 1,0000 | 0,7475 | 0,6389 | 1,0000 | Dinilai |
| 4 | BACH | Apa ancaman bisnis yang disebut paling utama bagi BACH? | 1,0000 | 0,6832 | 1,0000 | 1,0000 | Dinilai |
| 5 | BACH | Berapa total aset konsolidasian BACH per 31 Desember 2025 dalam Rupiah? | 0,0000 | 0,0000 | 0,0000 | 0,0000 | Dinilai |
| 6 | BACH | Berapa pendapatan neto konsolidasian BACH untuk tahun 2025 dalam Rupiah? | 1,0000 | 0,9743 | 0,5556 | 1,0000 | Dinilai |
| 7 | BACH | Jelaskan bidang usaha utama BACH dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | 0,5000 | 0,0000 | 0,0000 | 0,0000 | Dinilai |
| 8 | BACH | Apa risiko utama BACH, dan berapa pendapatan neto tahun 2025 dalam Rupiah pada laporan konsolidasian? | 0,6667 | 0,9073 | 0,0000 | 0,5000 | Dinilai |
| 9 | EMMI | Berapa harga penawaran saham EMMI pada prospektus ini? | 1,0000 | 0,9139 | 0,8056 | 1,0000 | Dinilai |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek IPO EMMI? | 0,5000 | 0,9255 | 0,9167 | 1,0000 | Dinilai |
| 11 | EMMI | Bisnis utama EMMI bergerak di bidang apa? | 1,0000 | 0,3862 | 0,8767 | 1,0000 | Dinilai |
| 12 | EMMI | Apa ancaman bisnis yang disebut paling utama bagi EMMI? | 1,0000 | 0,6828 | 0,6389 | 1,0000 | Dinilai |
| 13 | EMMI | Berapa total aset konsolidasian EMMI per 31 Desember 2025 dalam Rupiah? | 0,0000 | 0,0000 | 0,7500 | 1,0000 | Dinilai |
| 14 | EMMI | Berapa penjualan bersih konsolidasian EMMI untuk tahun 2025 dalam Rupiah? | 1,0000 | 0,9546 | 0,9167 | 1,0000 | Dinilai |
| 15 | EMMI | Jelaskan bidang usaha utama EMMI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | 1,0000 | 0,0000 | 0,0000 | 0,5000 | Dinilai |
| 16 | EMMI | Apa risiko utama EMMI, dan berapa penjualan bersih tahun 2025 dalam Rupiah pada laporan konsolidasian? | 1,0000 | 0,7349 | 0,0000 | 1,0000 | Dinilai |
| 17 | JECX | Berapa harga penawaran saham JECX pada prospektus ini? | 0,5000 | 0,9384 | 0,7500 | 1,0000 | Dinilai |
| 18 | JECX | Siapa penjamin pelaksana emisi efek IPO JECX? | 0,0000 | 0,9533 | 0,8333 | 1,0000 | Dinilai |
| 19 | JECX | Bisnis utama JECX bergerak di bidang apa? | 1,0000 | 0,4296 | 0,9500 | 1,0000 | Dinilai |
| 20 | JECX | Apa ancaman bisnis yang disebut paling utama bagi JECX? | 1,0000 | 0,5896 | 1,0000 | 1,0000 | Dinilai |
| 21 | JECX | Berapa total aset konsolidasian JECX per 31 Desember 2025 dalam jutaan Rupiah? | 0,0000 | 0,0000 | 0,3333 | 1,0000 | Dinilai |
| 22 | JECX | Berapa pendapatan konsolidasian JECX untuk tahun 2025 dalam jutaan Rupiah? | 0,0000 | 0,9717 | 0,6667 | 1,0000 | Dinilai |
| 23 | JECX | Jelaskan bidang usaha utama JECX dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | 1,0000 | 0,0000 | 0,0000 | 0,0000 | Dinilai |
| 24 | JECX | Apa risiko utama JECX, dan berapa pendapatan tahun 2025 dalam jutaan Rupiah pada laporan konsolidasian? | 0,6667 | 0,8806 | 0,0000 | 0,5000 | Dinilai |
| 25 | JELI | Berapa harga penawaran saham JELI pada prospektus ini? | 1,0000 | 0,9156 | 1,0000 | 1,0000 | Dinilai |
| 26 | JELI | Siapa penjamin pelaksana emisi efek IPO JELI? | 0,0000 | 0,9413 | 0,6792 | 1,0000 | Dinilai |
| 27 | JELI | Bisnis utama JELI bergerak di bidang apa? | 1,0000 | 0,4167 | 1,0000 | 1,0000 | Dinilai |
| 28 | JELI | Apa ancaman bisnis yang disebut paling utama bagi JELI? | 1,0000 | 0,7001 | 0,3333 | 1,0000 | Dinilai |
| 29 | JELI | Berapa total aset konsolidasian JELI per 31 Desember 2025 dalam Rupiah? | 1,0000 | 0,9670 | 0,2000 | 1,0000 | Dinilai |
| 30 | JELI | Berapa penjualan konsolidasian JELI untuk tahun 2025 dalam Rupiah? | 1,0000 | 0,9658 | 0,8333 | 1,0000 | Dinilai |
| 31 | JELI | Jelaskan bidang usaha utama JELI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | 0,6667 | 0,0000 | 0,3333 | 0,3333 | Dinilai |
| 32 | JELI | Apa risiko utama JELI, dan berapa penjualan tahun 2025 dalam Rupiah pada laporan konsolidasian? | 0,7500 | 0,7315 | 0,0000 | 1,0000 | Dinilai |
| 33 | PRDL | Berapa harga penawaran saham PRDL pada prospektus ini? | 1,0000 | 0,6856 | 0,5000 | 1,0000 | Dinilai |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek IPO PRDL? | 1,0000 | 0,9320 | 0,5250 | 1,0000 | Dinilai |
| 35 | PRDL | Bisnis utama PRDL bergerak di bidang apa? | 1,0000 | 0,9661 | 0,5667 | 1,0000 | Dinilai |
| 36 | PRDL | Apa ancaman bisnis yang disebut paling utama bagi PRDL? | 0,0000 | 0,9812 | 0,2500 | 1,0000 | Dinilai |
| 37 | PRDL | Berapa total aset Perseroan PRDL per 31 Desember 2025 dalam ribuan Rupiah? | 1,0000 | 0,9798 | 1,0000 | 1,0000 | Dinilai |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun 2025 dalam ribuan Rupiah? | 1,0000 | 0,9795 | 0,8333 | 1,0000 | Dinilai |
| 39 | PRDL | Jelaskan bidang usaha utama PRDL dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | 1,0000 | 0,0000 | 0,0000 | 0,0000 | Dinilai |
| 40 | PRDL | Apa risiko utama PRDL, dan berapa pendapatan dari kontrak dengan pelanggan tahun 2025 dalam ribuan Rupiah pada laporan Perseroan? | 0,5000 | 0,6761 | 0,0000 | 0,5000 | Dinilai |
| 41 | RANS | Berapa harga penawaran saham RANS pada prospektus ini? | 1,0000 | 0,8785 | 0,8667 | 1,0000 | Dinilai |
| 42 | RANS | Siapa penjamin pelaksana emisi efek IPO RANS? | 1,0000 | 0,9454 | 0,8667 | 1,0000 | Dinilai |
| 43 | RANS | Bisnis utama RANS bergerak di bidang apa? | 1,0000 | 0,4789 | 0,4778 | 1,0000 | Dinilai |
| 44 | RANS | Apa ancaman bisnis yang disebut paling utama bagi RANS? | 0,0000 | 0,0000 | 0,0000 | 0,0000 | Dinilai |
| 45 | RANS | Berapa total aset konsolidasian RANS per 31 Desember 2025 dalam Rupiah? | 0,0000 | 0,0000 | 0,0000 | 0,0000 | Dinilai |
| 46 | RANS | Berapa pendapatan konsolidasian RANS untuk tahun 2025 dalam Rupiah? | 1,0000 | 0,9715 | 0,1667 | 1,0000 | Dinilai |
| 47 | RANS | Jelaskan bidang usaha utama RANS dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | 0,6667 | 0,4986 | 0,0000 | 0,0000 | Dinilai |
| 48 | RANS | Apa risiko utama RANS, dan berapa pendapatan tahun 2025 dalam Rupiah pada laporan konsolidasian? | 1,0000 | 0,8591 | 0,0000 | 0,5000 | Dinilai |
| 49 | SWAP | Berapa harga penawaran saham SWAP pada prospektus ini? | 0,0000 | 0,9000 | 0,9500 | 1,0000 | Dinilai |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek IPO SWAP? | 1,0000 | 0,9407 | 0,9667 | 1,0000 | Dinilai |
| 51 | SWAP | Bisnis utama SWAP bergerak di bidang apa? | 0,5000 | 0,9623 | 1,0000 | 1,0000 | Dinilai |
| 52 | SWAP | Apa ancaman bisnis yang disebut paling utama bagi SWAP? | 1,0000 | 0,6866 | 0,5000 | 1,0000 | Dinilai |
| 53 | SWAP | Berapa total aset Perseroan SWAP per 31 Desember 2025 dalam Rupiah? | 1,0000 | 0,9638 | 1,0000 | 1,0000 | Dinilai |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun 2025 dalam Rupiah? | 0,0000 | 0,9346 | 0,5000 | 1,0000 | Dinilai |
| 55 | SWAP | Jelaskan bidang usaha utama SWAP dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | 1,0000 | 0,0000 | 0,0000 | 0,0000 | Dinilai |
| 56 | SWAP | Apa risiko utama SWAP, dan berapa pendapatan usaha tahun 2025 dalam Rupiah pada laporan Perseroan? | 1,0000 | 0,8145 | 0,0000 | 1,0000 | Dinilai |

#### Jawaban Sistem dan Jawaban Acuan

| No | Emiten | Pertanyaan | Jawaban sistem | Jawaban acuan | Keterangan |
| --- | --- | --- | --- | --- | --- |
| 1 | BACH | Berapa harga penawaran saham BACH pada prospektus ini? | Harga Penawaran saham BACH adalah Rp442,-. | Rp442 per saham. |  |
| 2 | BACH | Siapa penjamin pelaksana emisi efek IPO BACH? | Penjamin Pelaksana Emisi Efek IPO BACH adalah PT Erdikha Elit Sekuritas. | PT Erdikha Elit Sekuritas. |  |
| 3 | BACH | Bisnis utama BACH bergerak di bidang apa? | Bisnis utama PT BACH Multi Global Tbk bergerak di bidang penjualan dan penyewaan genset serta jasa konstruksi dan pemeliharaan infrastruktur telekomunikasi. | Penjualan dan penyewaan genset serta jasa konstruksi dan pemeliharaan infrastruktur telekomunikasi. |  |
| 4 | BACH | Apa ancaman bisnis yang disebut paling utama bagi BACH? | Ancaman bisnis yang disebut paling utama bagi Perseroan adalah risiko ketergantungan terhadap pihak pemasok genset. | Ketergantungan terhadap pihak pemasok genset. |  |
| 5 | BACH | Berapa total aset konsolidasian BACH per 31 Desember 2025 dalam Rupiah? | Informasi tersebut tidak tersedia di dokumen prospektus ini. | 1.232.630.823.230 Rupiah pada 31 Desember 2025. |  |
| 6 | BACH | Berapa pendapatan neto konsolidasian BACH untuk tahun 2025 dalam Rupiah? | Pendapatan neto konsolidasian BACH untuk tahun 2025 adalah Rp1.732.944.872.499. | 1.732.944.872.499 Rupiah untuk tahun 2025. |  |
| 7 | BACH | Jelaskan bidang usaha utama BACH dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Informasi mengenai bidang usaha utama BACH dan rincian alokasi dana IPO beserta nilai atau persentasenya tidak tersedia di dokumen prospektus ini. | Penjualan dan penyewaan genset serta jasa konstruksi dan pemeliharaan infrastruktur telekomunikasi. Rp91.020.000.000 untuk pembayaran sebagian utang kepada PT Bank Permata Tbk atas fasilitas Omnibus Revolving Loan. |  |
| 8 | BACH | Apa risiko utama BACH, dan berapa pendapatan neto tahun 2025 dalam Rupiah pada laporan konsolidasian? | Risiko utama BACH adalah risiko fluktuasi kurs mata uang asing akibat sebagian besar genset yang didistribusikan merupakan impor dengan menggunakan mata uang Dolar sementara seluruh pendapatan Perseroan diperoleh dalam mata uang Rupiah. Pendapatan neto tahun 2025 pada laporan konsolidasian sebesar Rp1.732.944.872.499. | Ketergantungan terhadap pihak pemasok genset. 1.732.944.872.499 Rupiah untuk tahun 2025. |  |
| 9 | EMMI | Berapa harga penawaran saham EMMI pada prospektus ini? | Harga penawaran saham EMMI adalah Rp470,- per saham. | Rp470 per saham. |  |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek IPO EMMI? | Penjamin Pelaksana Emisi Efek IPO EMMI adalah PT BRI Danareksa Sekuritas dan PT INA Sekuritas Indonesia. | PT BRI Danareksa Sekuritas dan PT INA Sekuritas Indonesia. |  |
| 11 | EMMI | Bisnis utama EMMI bergerak di bidang apa? | Bisnis utama Perseroan bergerak dalam perdagangan besar alat laboratorium, alat farmasi dan alat kedokteran untuk manusia. | Perdagangan besar alat laboratorium, alat farmasi, dan alat kedokteran untuk manusia. |  |
| 12 | EMMI | Apa ancaman bisnis yang disebut paling utama bagi EMMI? | Ancaman bisnis yang disebut paling utama bagi Perseroan adalah Risiko Konsentrasi Pelanggan, di mana sebagian besar penjualan ditujukan kepada instansi pemerintah sehingga terdapat risiko penurunan pendapatan jika terjadi pengurangan anggaran belanja kesehatan pemerintah. | Konsentrasi pelanggan. |  |
| 13 | EMMI | Berapa total aset konsolidasian EMMI per 31 Desember 2025 dalam Rupiah? | Informasi tersebut tidak tersedia di dokumen prospektus ini. | 564.439.161.798 Rupiah pada 31 Desember 2025. |  |
| 14 | EMMI | Berapa penjualan bersih konsolidasian EMMI untuk tahun 2025 dalam Rupiah? | Penjualan bersih konsolidasian EMMI untuk tahun 2025 sebesar Rp454.636.079.798. | 454.636.079.798 Rupiah untuk tahun 2025. |  |
| 15 | EMMI | Jelaskan bidang usaha utama EMMI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Bidang usaha utama EMMI tidak ditemukan dalam sumber yang tersedia. Satu rincian alokasi dana IPO adalah pembayaran sebagian pokok pinjaman Perseroan sebesar Rp50.000.000.000,- (lima puluh miliar Rupiah) untuk Perjanjian Kredit Bank Ina Perdana no. 405/BIP-ARB/KMK-TBH/XII/2024 juncto Akta Addendum Perjanjian Kredit No. 72 tanggal 17 Maret 2026. | Perdagangan besar alat laboratorium, alat farmasi, dan alat kedokteran untuk manusia. Rp50.000.000.000 untuk pembayaran sebagian pokok pinjaman Perseroan kepada Bank INA. |  |
| 16 | EMMI | Apa risiko utama EMMI, dan berapa penjualan bersih tahun 2025 dalam Rupiah pada laporan konsolidasian? | Risiko utama Perseroan adalah risiko konsentrasi pelanggan. Penjualan bersih tahun 2025 sebesar Rp454.636.079.798. | Konsentrasi pelanggan. 454.636.079.798 Rupiah untuk tahun 2025. |  |
| 17 | JECX | Berapa harga penawaran saham JECX pada prospektus ini? | Harga Penawaran Saham JECX ditetapkan sebesar Rp1.250 (seribu dua ratus lima puluh Rupiah) per saham. | Rp1.250 per saham. |  |
| 18 | JECX | Siapa penjamin pelaksana emisi efek IPO JECX? | Penjamin Pelaksana Emisi Efek IPO JECX adalah PT Trimegah Sekuritas Indonesia Tbk. | PT Trimegah Sekuritas Indonesia Tbk. |  |
| 19 | JECX | Bisnis utama JECX bergerak di bidang apa? | Bisnis utama Perseroan bergerak dalam bidang aktivitas rumah sakit swasta dan aktivitas klinik swasta. | Aktivitas rumah sakit swasta dan aktivitas klinik swasta. |  |
| 20 | JECX | Apa ancaman bisnis yang disebut paling utama bagi JECX? | Risiko utama yang dihadapi Perseroan adalah risiko ketergantungan Grup Perseroan terhadap para dokter, perawat, dan tenaga profesional yang berperan penting dalam melaksanakan strategi pertumbuhan maupun kelancaran operasional Grup Perseroan. | Ketergantungan Grup Perseroan terhadap dokter, perawat, dan tenaga profesional. |  |
| 21 | JECX | Berapa total aset konsolidasian JECX per 31 Desember 2025 dalam jutaan Rupiah? | Informasi tersebut tidak tersedia di dokumen prospektus ini. | 1.451.329 jutaan Rupiah pada 31 Desember 2025. |  |
| 22 | JECX | Berapa pendapatan konsolidasian JECX untuk tahun 2025 dalam jutaan Rupiah? | Pendapatan konsolidasian JECX untuk tahun 2025 adalah Rp926.763 juta. | 926.763 jutaan Rupiah untuk tahun 2025. |  |
| 23 | JECX | Jelaskan bidang usaha utama JECX dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Informasi mengenai bidang usaha utama JECX dan rincian alokasi dana IPO beserta nilai atau persentasenya tidak tersedia dalam kutipan sumber yang diberikan. | Aktivitas rumah sakit swasta dan aktivitas klinik swasta. Rp40.000.000.000 untuk pembayaran lebih awal atas sebagian pokok pinjaman Perseroan kepada PT Bank Central Asia Tbk. |  |
| 24 | JECX | Apa risiko utama JECX, dan berapa pendapatan tahun 2025 dalam jutaan Rupiah pada laporan konsolidasian? | Risiko utama JECX tidak disebutkan secara rinci dalam kutipan yang tersedia, namun dokumen menyebutkan Grup Perseroan menerapkan manajemen risiko untuk memitigasi faktor risiko yang dijelaskan pada Bab VI dengan judul “Faktor Risiko”. Pendapatan tahun 2025 dalam laporan keuangan konsolidasian adalah Rp926.763 juta. | Ketergantungan Grup Perseroan terhadap dokter, perawat, dan tenaga profesional. 926.763 jutaan Rupiah untuk tahun 2025. |  |
| 25 | JELI | Berapa harga penawaran saham JELI pada prospektus ini? | Harga penawaran saham JELI adalah Rp 900,- (sembilan ratus Rupiah) per lembar saham. | Rp900 per saham. |  |
| 26 | JELI | Siapa penjamin pelaksana emisi efek IPO JELI? | Penjamin Pelaksana Emisi Efek IPO JELI adalah PT Sucor Sekuritas. | PT Sucor Sekuritas. |  |
| 27 | JELI | Bisnis utama JELI bergerak di bidang apa? | Bisnis utama Perseroan bergerak dalam bidang industri makanan dan minuman, khususnya produksi dan perdagangan bahan makanan jelly, minuman sari kelapa, dan aloe vera, serta aktivitas pengepakan. | Industri makanan dan minuman penutup. |  |
| 28 | JELI | Apa ancaman bisnis yang disebut paling utama bagi JELI? | Ancaman bisnis yang disebut paling utama bagi Perseroan adalah risiko terkait dengan ketersediaan dan volatilitas harga bahan baku. | Ketersediaan dan volatilitas harga bahan baku. |  |
| 29 | JELI | Berapa total aset konsolidasian JELI per 31 Desember 2025 dalam Rupiah? | Total aset konsolidasian JELI per 31 Desember 2025 adalah Rp552.114.462.075. | 552.114.462.075 Rupiah pada 31 Desember 2025. |  |
| 30 | JELI | Berapa penjualan konsolidasian JELI untuk tahun 2025 dalam Rupiah? | Penjualan konsolidasian JELI untuk tahun 2025 adalah Rp753.050.908.792. | 753.050.908.792 Rupiah untuk tahun 2025. |  |
| 31 | JELI | Jelaskan bidang usaha utama JELI dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Bidang usaha utama JELI tidak ditemukan dalam sumber yang tersedia. Satu rincian alokasi dana IPO adalah penambahan penyertaan modal pada Perusahaan Anak, yaitu NPS, sebesar 56,70%. | Industri makanan dan minuman penutup. 56,70% untuk menambah penyertaan modal pada perusahaan anak NPS dalam bentuk ekuitas; NPS menggunakannya untuk belanja modal mesin dan peralatan produksi. |  |
| 32 | JELI | Apa risiko utama JELI, dan berapa penjualan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Risiko utama Perseroan adalah risiko harga komoditas, khususnya kenaikan harga gula yang merupakan bahan baku utama dan dapat memengaruhi marjin laba penjualan barang jadi. Penjualan tahun 2025 dalam laporan konsolidasian sebesar Rp753.050.908.792. | Ketersediaan dan volatilitas harga bahan baku. 753.050.908.792 Rupiah untuk tahun 2025. |  |
| 33 | PRDL | Berapa harga penawaran saham PRDL pada prospektus ini? | Harga Penawaran Saham Perseroan adalah sebesar Rp120,- (seratus dua puluh Rupiah) per lembar saham. | Rp120 per saham. |  |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek IPO PRDL? | Penjamin Pelaksana Emisi Efek IPO PRDL adalah PT Sucor Sekuritas. | PT Sucor Sekuritas. |  |
| 35 | PRDL | Bisnis utama PRDL bergerak di bidang apa? | Bisnis utama PRDL bergerak di bidang pembuatan dan pengolahan alat kesehatan terkait diagnosa medis. | Pembuatan dan pengolahan alat kesehatan terkait diagnosa medis. |  |
| 36 | PRDL | Apa ancaman bisnis yang disebut paling utama bagi PRDL? | Ancaman bisnis yang disebut paling utama bagi PRDL adalah risiko ketergantungan terhadap belanja Pemerintah pada Sektor Kesehatan. | Ketergantungan terhadap belanja pemerintah pada sektor kesehatan. |  |
| 37 | PRDL | Berapa total aset Perseroan PRDL per 31 Desember 2025 dalam ribuan Rupiah? | Total aset Perseroan PRDL per 31 Desember 2025 adalah Rp194.428.445 ribu. | 194.428.445 ribuan Rupiah pada 31 Desember 2025. |  |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun 2025 dalam ribuan Rupiah? | Pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun 2025 adalah Rp74.371.107 ribu. | 74.371.107 ribuan Rupiah untuk tahun 2025. |  |
| 39 | PRDL | Jelaskan bidang usaha utama PRDL dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Bidang usaha utama Perseroan belum ditemukan dalam sumber yang tersedia. Satu rincian alokasi dana IPO adalah sekitar 10,18% untuk modal kerja Perseroan termasuk pembelian bahan baku, biaya product & development dan selling & marketing. | Pembuatan dan pengolahan alat kesehatan terkait diagnosa medis. Rp35.666.666.667 untuk pelunasan pokok fasilitas kredit kepada PT Bank Central Asia Tbk dan PT Bank Pan Indonesia Tbk. |  |
| 40 | PRDL | Apa risiko utama PRDL, dan berapa pendapatan dari kontrak dengan pelanggan tahun 2025 dalam ribuan Rupiah pada laporan Perseroan? | Informasi mengenai risiko utama PRDL tidak tersedia di dokumen prospektus ini. Pendapatan dari kontrak dengan pelanggan tahun 2025 adalah Rp74.371.107 ribu. | Ketergantungan terhadap belanja pemerintah pada sektor kesehatan. 74.371.107 ribuan Rupiah untuk tahun 2025. |  |
| 41 | RANS | Berapa harga penawaran saham RANS pada prospektus ini? | Harga Penawaran saham PT RANS Entertainment Indonesia Tbk adalah Rp170,- (seratus tujuh puluh Rupiah) setiap saham. | Rp170 per saham. |  |
| 42 | RANS | Siapa penjamin pelaksana emisi efek IPO RANS? | Penjamin Pelaksana Emisi Efek IPO RANS adalah PT Trimegah Sekuritas Indonesia Tbk. | PT Trimegah Sekuritas Indonesia Tbk. |  |
| 43 | RANS | Bisnis utama RANS bergerak di bidang apa? | Bisnis utama Perseroan bergerak dalam bidang usaha media dan hiburan, meliputi produksi dan distribusi konten, pengelolaan intellectual property (IP), penyelenggaraan event dan seni pertunjukan, periklanan, aktivitas taman bertema dan taman hiburan, arena permainan, serta aktivitas perusahaan holding. | Media dan hiburan, termasuk produksi dan distribusi konten, pengelolaan intellectual property, event, periklanan, taman hiburan, arena permainan, dan aktivitas holding. |  |
| 44 | RANS | Apa ancaman bisnis yang disebut paling utama bagi RANS? | Informasi mengenai ancaman bisnis yang disebut paling utama bagi RANS tidak tersedia di dokumen prospektus ini. | Ketergantungan terhadap talent utama Raffi Ahmad dan Nagita Slavina beserta keluarga. |  |
| 45 | RANS | Berapa total aset konsolidasian RANS per 31 Desember 2025 dalam Rupiah? | Informasi tersebut tidak tersedia di dokumen prospektus ini. | 461.034.721.642 Rupiah pada 31 Desember 2025. |  |
| 46 | RANS | Berapa pendapatan konsolidasian RANS untuk tahun 2025 dalam Rupiah? | Pendapatan konsolidasian RANS untuk tahun 2025 adalah sebesar Rp 353.377.040.895. | 353.377.040.895 Rupiah untuk tahun 2025. |  |
| 47 | RANS | Jelaskan bidang usaha utama RANS dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Bidang usaha utama Perseroan adalah RANS Entertainment Indonesia, Tbk. Salah satu rincian alokasi dana IPO adalah untuk melakukan pembayaran lebih awal (pelunasan dipercepat) atas seluruh pokok utang kepada PT Bank Negara Indonesia (Persero) Tbk dengan jumlah sekitar 6,98% atau sekitar Rp.. | Media dan hiburan, termasuk produksi dan distribusi konten, pengelolaan intellectual property, event, periklanan, taman hiburan, arena permainan, dan aktivitas holding. Sekitar Rp29.950.000.000 untuk pembayaran lebih awal atas seluruh pokok utang kepada PT Bank Negara Indonesia (Persero) Tbk. |  |
| 48 | RANS | Apa risiko utama RANS, dan berapa pendapatan tahun 2025 dalam Rupiah pada laporan konsolidasian? | Risiko utama RANS adalah ketergantungan terhadap talent utama yaitu Raffi Ahmad dan Nagita Slavina beserta keluarga. Pendapatan tahun 2025 tidak dapat dipastikan dari dokumen karena sumber hanya menyajikan Laba (Rugi) Tahun Berjalan yang dapat diatribusikan kepada pemilik perusahaan sebesar Rp61.049.565.722, bukan pendapatan. | Ketergantungan terhadap talent utama Raffi Ahmad dan Nagita Slavina beserta keluarga. 353.377.040.895 Rupiah untuk tahun 2025. |  |
| 49 | SWAP | Berapa harga penawaran saham SWAP pada prospektus ini? | Harga penawaran saham SWAP ditetapkan sebesar Rp130 hingga Rp140 per saham. | Rp130 sampai dengan Rp140 per saham, berupa rentang pada prospektus awal. |  |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek IPO SWAP? | Penjamin Pelaksana Emisi Efek IPO SWAP adalah PT Sukadana Prima Sekuritas. | PT Sukadana Prima Sekuritas. |  |
| 51 | SWAP | Bisnis utama SWAP bergerak di bidang apa? | Bisnis utama SWAP bergerak dalam bidang industri manufaktur produk kesehatan berbasis riset, dengan kegiatan usaha utama meliputi industri sabun dan bahan pembersih keperluan rumah tangga. | Industri manufaktur produk kesehatan berbasis riset. |  |
| 52 | SWAP | Apa ancaman bisnis yang disebut paling utama bagi SWAP? | Ancaman bisnis yang disebut paling utama bagi Perseroan adalah risiko dalam distribusi dan pemasaran produk karena produk Perseroan adalah produk hasil penelitian yang membutuhkan edukasi dan penetrasi pasar. | Distribusi dan pemasaran. |  |
| 53 | SWAP | Berapa total aset Perseroan SWAP per 31 Desember 2025 dalam Rupiah? | Total aset Perseroan SWAP per 31 Desember 2025 adalah sebesar Rp33.362.492.250. | 33.362.492.250 Rupiah pada 31 Desember 2025. |  |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun 2025 dalam Rupiah? | Pendapatan usaha Perseroan SWAP untuk tahun yang berakhir pada tanggal 31 Desember 2025 adalah sebesar Rp7.121.355.527. | 7.121.355.527 Rupiah untuk tahun 2025. |  |
| 55 | SWAP | Jelaskan bidang usaha utama SWAP dan satu rincian alokasi dana IPO yang dicantumkan dalam rencana penggunaan dana, beserta nilai atau persentasenya. | Bidang usaha utama Perseroan tidak dapat dipastikan dari dokumen yang tersedia. Rincian alokasi dana IPO adalah sekitar Rp 15.800.000.000,- (lima belas miliar delapan ratus juta Rupiah) untuk modal kerja dengan tujuan meningkatkan kapasitas produksi dan kualitas produk. | Industri manufaktur produk kesehatan berbasis riset. Sekitar Rp500.000.000 untuk pembelian aset gedung baru berupa furniture, AC, interior, serta instalasi. |  |
| 56 | SWAP | Apa risiko utama SWAP, dan berapa pendapatan usaha tahun 2025 dalam Rupiah pada laporan Perseroan? | Risiko utama Perseroan adalah risiko distribusi dan pemasaran, dengan pendapatan usaha tahun 2025 sebesar Rp7.121.355.527. | Distribusi dan pemasaran. 7.121.355.527 Rupiah untuk tahun 2025. |  |

## Simpan Hasil

In [9]:
tables = {
    "L12_1_distribusi.csv": distribusi,
    "L12_2_retrieval_keseluruhan.csv": keseluruhan,
    "L12_3_retrieval_kategori.csv": kategori,
    "retrieval_per_pertanyaan.csv": rincian_retrieval,
    "retrieval_peringkat_chunk.csv": peringkat_retrieval,
    "ragas_ringkasan.csv": ringkasan_ragas,
    "ragas_per_pertanyaan.csv": rincian_ragas,
    "ragas_konteks_per_pertanyaan.csv": konteks_ragas,
}
for filename, rows in tables.items():
    write_csv(HASIL / filename, rows)

cakupan = (
    f"{len(rincian_ragas)} pertanyaan dengan jawaban diuji. "
    f"{status_ragas.get('scored', 0)} pertanyaan mendapat empat skor RAGAS. "
    f"{jumlah_tanpa_konteks} pertanyaan tidak membentuk konteks. "
    f"Cakupan penilaian: {cakupan_teks}%."
)
report = "\n\n".join([
    "# Hasil Pengujian Retrieval",
    "## Tabel L12.1 Distribusi Dokumen dan Pertanyaan Pengujian Retrieval",
    markdown_table(distribusi),
    "## Tabel L12.2 Hasil Pengujian Retrieval Keseluruhan",
    markdown_table(keseluruhan),
    "## Tabel L12.3 Hasil Retrieval Berdasarkan Kategori Pertanyaan pada K = 6",
    markdown_table(kategori),
    "## Hasil Pengujian Retrieval per Pertanyaan",
    markdown_table(tabel_retrieval),
    "Peringkat dan teks chunk lengkap tersedia dalam retrieval_peringkat_chunk.csv.",
    "# Hasil Pengujian RAGAS",
    markdown_table(ringkasan_ragas),
    cakupan,
    protokol_teks,
    "## Hasil Pengujian RAGAS per Pertanyaan",
    "### Skor RAGAS per Pertanyaan",
    markdown_table(tabel_skor_ragas),
    "### Jawaban Sistem dan Jawaban Acuan",
    markdown_table(tabel_jawaban_ragas),
    "Konteks yang dikirim ke model tersedia dalam ragas_konteks_per_pertanyaan.csv.",
])
(HASIL / "Hasil_Evaluasi.md").write_text(report + "\n", encoding="utf-8")
for filename in ("Hasil_Evaluasi.md", *tables):
    relative_path = Path(os.path.relpath(HASIL / filename, Path.cwd())).as_posix()
    display(Markdown(f"[{filename}](<{relative_path}>)"))


[Hasil_Evaluasi.md](<evaluation/results/evaluasi_retrieval_ragas/Hasil_Evaluasi.md>)

[L12_1_distribusi.csv](<evaluation/results/evaluasi_retrieval_ragas/L12_1_distribusi.csv>)

[L12_2_retrieval_keseluruhan.csv](<evaluation/results/evaluasi_retrieval_ragas/L12_2_retrieval_keseluruhan.csv>)

[L12_3_retrieval_kategori.csv](<evaluation/results/evaluasi_retrieval_ragas/L12_3_retrieval_kategori.csv>)

[retrieval_per_pertanyaan.csv](<evaluation/results/evaluasi_retrieval_ragas/retrieval_per_pertanyaan.csv>)

[retrieval_peringkat_chunk.csv](<evaluation/results/evaluasi_retrieval_ragas/retrieval_peringkat_chunk.csv>)

[ragas_ringkasan.csv](<evaluation/results/evaluasi_retrieval_ragas/ragas_ringkasan.csv>)

[ragas_per_pertanyaan.csv](<evaluation/results/evaluasi_retrieval_ragas/ragas_per_pertanyaan.csv>)

[ragas_konteks_per_pertanyaan.csv](<evaluation/results/evaluasi_retrieval_ragas/ragas_konteks_per_pertanyaan.csv>)